# 28. 통합 holdout 후보 실행 전 봉인

이 노트북은 **Codex coder agent**가 작성·실행한 CPU/offline preflight다. 결과를 보지 않은 상태에서 OLD/STRUCT 두 후보의 실행 계약과 단계 순서를 봉인하고, 30개 query-only embedding 승인 범위를 계산한다. gold·qualification·ranking·BGE·LLM 평가는 이 실행에서 수행하지 않는다.


## 결과 전 고정 계약

실행 순서는 candidate/source seal → query-only freeze/cache lookup → 별도 사용자 승인 embedding → label-free ranking freeze → local BGE freeze → 그 뒤에만 sealed gold/qualification/scoring load → retrieval 평가 → LLM payload freeze/별도 승인 → response freeze → scoring이다. 현재 후보 실행 seal은 승인되지 않았으며, 외부 호출·GPU·검색·모델 로드는 모두 0이다.

27번의 `sealed_not_evaluated` dataset seal과 독립 reviewer의 `SEALABLE` 판정은 이 preflight를 준비할 근거다. scoring semantics/hash는 고정됐지만 별도 candidate execution seal은 사용자 승인 전까지 권한이 없다. 행 내부의 과거 `pending_independent_review` 표기는 lineage 값이며 dataset seal의 최종 상태를 소급 변경하지 않는다.


In [1]:
from pathlib import Path
import hashlib, json, math, os, platform, stat, unicodedata
from collections import Counter, defaultdict
import numpy as np
import tiktoken

cwd = Path.cwd().resolve()
if (cwd / 'notebooks').is_dir(): ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'notebooks').is_dir(): ROOT = cwd.parent
else: raise RuntimeError(f'fail-closed unexpected cwd: {cwd}')
NOTEBOOK = ROOT / 'notebooks/28_integrated_holdout_candidate_evaluation.ipynb'
OUT = ROOT / 'notebooks/data/28_integrated_holdout_candidate_evaluation'
OUT.mkdir(parents=True, exist_ok=True)
if os.environ.get('RUN_APPROVED_28_QUERY_EMBEDDING', '0') not in {'0','1'}: raise RuntimeError('invalid query embedding execution flag')
if os.environ.get('RUN_APPROVED_28_LOCAL_GPU', '0') not in {'0','1'}: raise RuntimeError('invalid local GPU execution flag')
if os.environ.get('RUN_APPROVED_28_POST_FREEZE', '0') not in {'0','1'}: raise RuntimeError('invalid post-freeze execution flag')

def canon(value): return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
def sha_bytes(value): return hashlib.sha256(value).hexdigest()
def sha(path): return sha_bytes(path.read_bytes())
def text_sha(value): return sha_bytes(value.encode('utf-8'))
def norm(value): return ' '.join(unicodedata.normalize('NFKC', str(value)).casefold().split())
def read_json(rel): return json.loads((ROOT / rel).read_text(encoding='utf-8'))
def atomic_write_text(path, value):
    tmp=path.with_name('.'+path.name+'.tmp'); tmp.write_text(value,encoding='utf-8'); os.replace(tmp,path); return path
def write_json(name, value):
    return atomic_write_text(OUT/name,json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2)+'\n')
def source_only_notebook_sha(path):
    doc = json.loads(path.read_text(encoding='utf-8'))
    core = [{'cell_type': c['cell_type'], 'id': c.get('id'), 'source': c.get('source', [])} for c in doc['cells']]
    return text_sha(canon(core))

OLD_CACHE_NAMES = [
 '3c81bda6bc8b1e69ca305b0dcc219203baeede6797e91045dbde36244babc3b4',
 '335a583c624bd1fe61aec74282af4e1462af20962c96147ada05ed5a29605fba',
 '93cd3f7a2ca8c9e070421705a575465a92f29966aebae38a583c2da60a6ee97f',
 'b6ae13c54bbba3d250e83c39da8673647b077dfc91f671565b4cef016593fbf2',
 '8433343533d6e8ae60f40c42d7e4a0b43b751bb36e26532e738ff51eac4a9ae0',
 'cb5d9e87e091be498d91fa51d46619ca60c7fe5314b27a6686b017cb8c304e55']
STRUCT_CACHE_NAMES = [
 'cae935c970ad91dd9f60f6ee9c8b60c8693fea7e3c075198a176093abc159582',
 '2a156daf369c8d96be05ac2b0861cb38cd3ac92193a3114ff83d6449d8f0049e',
 '40f6ee0b1c85ee2fa3dca1ac5e927c92decb6ed86c5ff26c7bf8845c6fbaf970']
QUERY_CACHE_RELS = [
 'notebooks/data/24_multicard_recommendation_retrieval_evaluation/embedding_cache/text-embedding-3-small/031928bee3ba51aa700afbbb6903e0a1e839a1afcfbe7a4e45dbdb5a8d4b2027.npz',
 'notebooks/data/24_multicard_recommendation_retrieval_evaluation/embedding_cache/text-embedding-3-small/fca33efe64c4dfe0f728c9c7e7eeb7d5ea9b61a48b571f75416bca1f2bc9454b.npz']
OLD_CACHE_RELS = ['notebooks/data/13_hierarchical_chunking_retrieval/embedding_cache/text-embedding-3-small/' + x + '.npz' for x in OLD_CACHE_NAMES]
STRUCT_CACHE_RELS = ['notebooks/data/22_structural_heading_chunking_ablation/embedding_cache/text-embedding-3-small/' + x + '.npz' for x in STRUCT_CACHE_NAMES]
MODEL_RELS = [
 '.cache/reranker/bge-reranker-v2-m3/config.json',
 '.cache/reranker/bge-reranker-v2-m3/model.safetensors',
 '.cache/reranker/bge-reranker-v2-m3/sentencepiece.bpe.model',
 '.cache/reranker/bge-reranker-v2-m3/special_tokens_map.json',
 '.cache/reranker/bge-reranker-v2-m3/tokenizer.json',
 '.cache/reranker/bge-reranker-v2-m3/tokenizer_config.json']
SOURCE_RELS = [
 'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl',
 'notebooks/data/13_hierarchical_chunking_retrieval/embedding_usage.json',
 'notebooks/data/13_hierarchical_chunking_retrieval/input_manifest.json',
 'notebooks/data/13_hierarchical_chunking_retrieval/index_manifest.json',
 'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json',
 'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/phase_a_classification_summary.json',
 'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl',
 'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl',
 'notebooks/data/22_structural_heading_chunking_ablation/embedding_usage.json',
 'notebooks/data/22_structural_heading_chunking_ablation/input_manifest.json',
 'notebooks/data/23_structural_chunking_reranker_comparison/evaluation_contract.json',
 'notebooks/data/23_structural_chunking_reranker_comparison/preflight.json',
 'notebooks/data/24_multicard_recommendation_retrieval_evaluation/followup1_contract.json',
 'notebooks/data/24_multicard_recommendation_retrieval_evaluation/followup1_bundle_build_freeze.json',
 'notebooks/data/27_integrated_holdout_dataset/dataset_seal.json',
 'notebooks/data/27_integrated_holdout_dataset/draft_dataset_manifest.json',
 'notebooks/data/27_integrated_holdout_dataset/evaluation_queries.json']
ALL_READ_ONLY_RELS = SOURCE_RELS + OLD_CACHE_RELS + STRUCT_CACHE_RELS + QUERY_CACHE_RELS + MODEL_RELS
if len(ALL_READ_ONLY_RELS) != len(set(ALL_READ_ONLY_RELS)): raise RuntimeError('duplicate input path')
for rel in ALL_READ_ONLY_RELS:
    if not (ROOT / rel).is_file(): raise RuntimeError(f'missing sealed input: {rel}')
input_hashes_before = {rel: sha(ROOT / rel) for rel in ALL_READ_ONLY_RELS}

dataset_seal = read_json('notebooks/data/27_integrated_holdout_dataset/dataset_seal.json')
seal_core = dataset_seal['seal_core']
if text_sha(canon(seal_core)) != dataset_seal['dataset_seal_sha256']: raise RuntimeError('dataset seal self hash mismatch')
if seal_core['status'] != 'sealed_not_evaluated' or seal_core['reviewer_provenance']['verdict'] != 'SEALABLE': raise RuntimeError('dataset is not sealed')
if seal_core['record_count'] != 30 or len(set(seal_core['record_ids'])) != 30: raise RuntimeError('record seal mismatch')
query_doc = read_json('notebooks/data/27_integrated_holdout_dataset/evaluation_queries.json')
if input_hashes_before['notebooks/data/27_integrated_holdout_dataset/evaluation_queries.json'] != seal_core['hashes']['query']: raise RuntimeError('sealed query hash mismatch')
records = query_doc['records']
if [r['record_id'] for r in records] != seal_core['record_ids']: raise RuntimeError('query order mismatch')

router_sha = 'e4e2c0b33dc228382e269a3edb399b1b5b4d0c4f8d41149e96d4d06d64d55909'
c21 = read_json('notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json')
c23 = read_json('notebooks/data/23_structural_chunking_reranker_comparison/evaluation_contract.json')
f1 = read_json('notebooks/data/24_multicard_recommendation_retrieval_evaluation/followup1_contract.json')
if c21['phase_a']['classifier_rule_sha256'] != router_sha or c23['classifier']['rule_sha256'] != router_sha: raise RuntimeError('router source mismatch')
if not (c21['models']['bge']['revision'] == c23['models']['bge']['revision'] == f1['bge']['revision']): raise RuntimeError('BGE revision mismatch')
if f1['relation_order'] != ['best_rrf_seed','same_node_adjacent_parts','non_root_immediate_parent_direct_body','heading_only_parent_context_and_same_parent_direct_body_children_by_heading_distance_then_earlier','seed_node_direct_children','same_card_other_top20_seeds_rrf_order']: raise RuntimeError('bundle rule drift')

model_hashes = {rel: input_hashes_before[rel] for rel in MODEL_RELS}
if model_hashes['.cache/reranker/bge-reranker-v2-m3/config.json'] != '13dcd6c31d9fec9d1d8e158702072f62d7fa7d312a64b9fe057bec9a08cfe41a': raise RuntimeError('BGE config drift')
if model_hashes['.cache/reranker/bge-reranker-v2-m3/model.safetensors'] != 'd9e3e081faff1eefb84019509b2f5558fd74c1a05a2c7db22f74174fcedb5286': raise RuntimeError('BGE weights drift')
if model_hashes['.cache/reranker/bge-reranker-v2-m3/tokenizer_config.json'] != '7e4c1cc848840aeccdd763458c18dd525eb0f795c992e00ebe9c28554e7db2d4': raise RuntimeError('BGE tokenizer config drift')
bge_fingerprint_sha = text_sha(canon(model_hashes))

common_search = {'embedding_model':'text-embedding-3-small','vector_rank_depth':50,'bm25_rank_depth':50,'vector_weight':0.4,'bm25_weight':0.6,'rrf_k':60,'bm25_k1':1.5,'bm25_b':0.75,'rrf_score':'0.4/(60+vector_rank)+0.6/(60+bm25_rank)','component_rank_limit':50,'fused_worklist_depth':50,'tie_break':['fused_score_desc','component_best_rank_asc','chunk_id_asc'],'raw_score_mixing':False,'output_unique_cards':3}
bge = {'revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','fingerprint_sha256':bge_fingerprint_sha,'model_file_hashes':model_hashes,'architecture':'XLMRobertaForSequenceClassification','batch_size':2,'dtype':'float16','max_length':8192,'truncation':'only_second','padding':'dynamic','local_files_only':True,'trust_remote_code':False,'score':'raw single logit desc','rrf_logit_mixing':False}
candidate_contract = {
 'schema_version':'integrated_holdout_candidate_preflight_v1','declared_before_results':True,'authorized':False,'status':'sealed_awaiting_query_embedding_approval',
 'execution_order':['candidate_config_and_source_hash_seal','query_only_freeze_and_hash_cache_lookup','user_approved_query_embedding','label_free_ranking_freeze','local_bge_freeze','sealed_gold_qualification_scoring_load','retrieval_evaluation','llm_payload_freeze_and_separate_approval','response_freeze','scoring'],
 'stage_access':{'current_stage':'query_only_offline_preflight','gold_master_qualification_read':0,'ranking':0,'retrieval':0,'bge_model_load':0,'bge_scoring':0,'gpu':0,'api':0,'network':0,'new_embedding':0,'llm':0},
 'dataset_authority':{'dataset_seal_sha256':dataset_seal['dataset_seal_sha256'],'generation_id':seal_core['generation_id'],'record_ids':seal_core['record_ids'],'reviewer_provenance':seal_core['reviewer_provenance'],'dataset_status':seal_core['status'],'scoring_sha256':seal_core['hashes']['scoring'],'interpretation':'dataset seal and independent SEALABLE provenance authorize preparation only; frozen scoring semantics do not authorize candidate execution before separate user approval'},
 'common_search':common_search,'router':{'input_allowlist':['query_text'],'rule_sha256':router_sha,'routes':{'proper_noun':'no_reranker','numeric_condition':'no_reranker','semantic':'reranker'}},
 'candidate_A_OLD':{'corpus':'327 old card/page/section/benefit chunks','ranking':'common fused Top50 then order-preserving level filter to section/benefit, then first 20','semantic_reranker_input':'safe augmented issuer > card_name > level > first Markdown heading plus document body','rerank_scope':'semantic queries only; raw BGE logit over the filtered D20 chunks','reranker_tie_break':['raw_logit_desc','original_rrf_rank_asc','chunk_id_asc'],'passthrough':'proper and numeric keep RRF order','card_collapse':'first occurrence per card, Top3','evidence_package':'representative then same-card remaining D20 evidence in final order, maximum 5 unique section/benefit chunks'},
 'candidate_B_STRUCT':{'corpus':'147 structural direct-body chunks','ranking':'common fused Top50 then first 20','rerank_scope':'all queries; one BGE score per card bundle','bundle_rule_source_sha256':input_hashes_before['notebooks/data/24_multicard_recommendation_retrieval_evaluation/followup1_contract.json'],'relation_order':f1['relation_order'],'bundle_limits':{'max_unique_chunks':5,'document_tokens':4096,'overflow':'package_failure','forbidden':f1['forbidden']},'card_ranking_tie_break':['raw_logit_desc','best_seed_rrf_rank_asc','card_key_asc'],'card_collapse':'BGE card ranking, Top3'},
 'bge':bge,'failure_policy':{'missing_query_embedding':'stop_before_ranking','bundle_overflow':'package_failure','model_or_cache_hash_mismatch':'invalid_execution','no_post_result_tuning':True},
 'later_data_access':{'gold_master_qualification':'only after ranking and local BGE freezes are sealed','llm_payload':'separate approval required','response_scoring':'only after response freeze'},
 'manual_discovery_incident':{'status':'retained_provenance','fact':'before this notebook run, one manual find command printed one file path under a prohibited prior evaluation directory','content_read':False,'hash_computed':False,'used_by_notebook_or_outputs':False,'affected_candidate_inputs':False}
}
candidate_core_sha = text_sha(canon(candidate_contract))
bge_root=ROOT/'.cache/reranker/bge-reranker-v2-m3'; bge_tree_paths=sorted(p for p in bge_root.rglob('*') if p.is_file()); bge_tree_files={str(p.relative_to(bge_root)):sha(p) for p in bge_tree_paths}; bge_tree_fingerprint=text_sha(canon(bge_tree_files))
execution_input_rels=['notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl','notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl','notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl']+OLD_CACHE_RELS+STRUCT_CACHE_RELS+MODEL_RELS
execution_input_hashes={rel:sha(ROOT/rel) for rel in execution_input_rels}
write_json('candidate_execution_contract.json', candidate_contract)
write_json('candidate_config_seal.json', {'candidate_core':candidate_contract,'candidate_core_sha256':candidate_core_sha,'source_hashes':{rel:input_hashes_before[rel] for rel in SOURCE_RELS},'execution_input_hashes':execution_input_hashes,'bge_cache':{'root':'.cache/reranker/bge-reranker-v2-m3','tree_file_count':len(bge_tree_files),'tree_total_bytes':sum(p.stat().st_size for p in bge_tree_paths),'tree_files':bge_tree_files,'tree_fingerprint_sha256':bge_tree_fingerprint,'required_model_files':model_hashes,'custom_code_required':False,'custom_required_files':[]},'status':'sealed_awaiting_query_embedding_approval','authorized':False})

# Query plaintext is confined to this restricted artifact. Public manifests below contain hashes/counts only.
enc = tiktoken.get_encoding('cl100k_base')
restricted = []
for row in records:
    text = row['sealed_query_text']; normalized = norm(text)
    restricted.append({'record_id':row['record_id'],'query_text':text,'raw_text_sha256':text_sha(text),'normalized_text_sha256':text_sha(normalized),'cl100k_tokens':len(enc.encode(text))})
restricted_path = write_json('restricted_query_freeze.json', {'schema_version':'restricted_query_freeze_v1','ordered_queries':restricted})
restricted_path.chmod(stat.S_IRUSR | stat.S_IWUSR)
if len(restricted) != 30 or len({x['raw_text_sha256'] for x in restricted}) != 30 or len({x['normalized_text_sha256'] for x in restricted}) != 30: raise RuntimeError('query uniqueness failure')
query_freeze_sha = sha(restricted_path)

def load_cache_rows(rels):
    rows=[]
    for cache_order, rel in enumerate(rels):
        with np.load(ROOT / rel, allow_pickle=False) as z:
            if 'hashes' not in z.files or 'embeddings' not in z.files: raise RuntimeError(f'bad cache schema {rel}')
            vectors=z['embeddings']; hashes=z['hashes']
            if vectors.dtype != np.float32 or vectors.ndim != 2 or vectors.shape[1] != 1536 or len(hashes) != len(vectors) or not np.isfinite(vectors).all(): raise RuntimeError(f'bad vectors {rel}')
            rows.extend((str(h), cache_order, i) for i,h in enumerate(hashes))
    return rows
old_cache_rows=load_cache_rows(OLD_CACHE_RELS); struct_cache_rows=load_cache_rows(STRUCT_CACHE_RELS); query_cache_rows=load_cache_rows(QUERY_CACHE_RELS)
cache_hashes=defaultdict(list)
for source, rows in [('old_document_cache',old_cache_rows),('struct_document_cache',struct_cache_rows),('prior_query_cache',query_cache_rows)]:
    for h, cache_order, row_order in rows: cache_hashes[h].append({'source':source,'cache_order':cache_order,'row_order':row_order})

old_chunks=[json.loads(x) for x in (ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').read_text(encoding='utf-8').splitlines() if x]
struct_chunks=[json.loads(x) for x in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl').read_text(encoding='utf-8').splitlines() if x]
if len(old_chunks)!=327 or len({x['id'] for x in old_chunks})!=327: raise RuntimeError('OLD corpus count')
if len(struct_chunks)!=147 or len({x['chunk_id'] for x in struct_chunks})!=147: raise RuntimeError('STRUCT corpus count')
old_doc_hits=sum(text_sha(x['document']) in {h for h,_,_ in old_cache_rows} for x in old_chunks)
struct_doc_hits=sum(text_sha(x['retrieval_text']) in {h for h,_,_ in struct_cache_rows} for x in struct_chunks)
if old_doc_hits != 327 or struct_doc_hits != 147: raise RuntimeError('document embedding coverage')

lookup=[]
for q in restricted:
    raw_matches=cache_hashes.get(q['raw_text_sha256'],[])
    norm_matches=cache_hashes.get(q['normalized_text_sha256'],[])
    lookup.append({'record_id':q['record_id'],'raw_text_sha256':q['raw_text_sha256'],'normalized_text_sha256':q['normalized_text_sha256'],'raw_cache_hit':bool(raw_matches),'normalized_cache_hit':bool(norm_matches),'raw_match_count':len(raw_matches),'normalized_match_count':len(norm_matches),'cl100k_tokens':q['cl100k_tokens']})
misses=[x for x in lookup if not x['raw_cache_hit']]
query_manifest={'schema_version':'query_embedding_manifest_v1','plaintext_included':False,'model':'text-embedding-3-small','query_count':30,'unique_raw_text_count':len({x['raw_text_sha256'] for x in lookup}),'unique_normalized_text_count':len({x['normalized_text_sha256'] for x in lookup}),'ordered_lookup':lookup,'raw_cache_hits':sum(x['raw_cache_hit'] for x in lookup),'raw_cache_misses':len(misses),'normalized_cache_hits':sum(x['normalized_cache_hit'] for x in lookup),'transmit_items':len(misses),'transmit_tokens':sum(x['cl100k_tokens'] for x in misses),'batch_size':64,'max_requests':math.ceil(len(misses)/64),'tokenizer':'cl100k_base','cost_usd':None,'cost_status':'unconfirmed_official_price_not_checked_in_this_offline_run'}
write_json('query_embedding_manifest.json',query_manifest)
approval_core={'schema_version':'query_embedding_approval_core_v1','candidate_core_sha256':candidate_core_sha,'dataset_seal_sha256':dataset_seal['dataset_seal_sha256'],'query_freeze_sha256':query_freeze_sha,'model':'text-embedding-3-small','ordered_miss_record_ids':[x['record_id'] for x in misses],'ordered_miss_raw_text_sha256':[x['raw_text_sha256'] for x in misses],'item_cap':len(misses),'token_cap_cl100k':sum(x['cl100k_tokens'] for x in misses),'request_cap':math.ceil(len(misses)/64),'batch_size':64,'transmitted_allowlist':['ordered query text only'],'transmitted_forbidden':['documents','chunks','gold','qualification','claims','expected cards','evidence spans','required terms','categories','metrics','metadata','secrets'],'authorized':False}
approval_sha=text_sha(canon(approval_core))
write_json('query_embedding_approval.json',{'approval_core':approval_core,'approval_core_sha256':approval_sha,'status':'awaiting_explicit_user_approval','authorized':False,'plaintext_artifact':'restricted_query_freeze.json (mode 0600)'})
write_json('document_embedding_audit.json',{'model':'text-embedding-3-small','dimension':1536,'dtype':'float32','finite':True,'authoritative_duplicate_policy':'embedding_usage batch order then row order; first occurrence; exact content hash only','OLD':{'chunks':327,'covered':old_doc_hits,'cache_rows':len(old_cache_rows),'cache_files':len(OLD_CACHE_RELS)},'STRUCT':{'chunks':147,'covered':struct_doc_hits,'cache_rows':len(struct_cache_rows),'cache_files':len(STRUCT_CACHE_RELS)},'bge':{'revision':bge['revision'],'required_file_count':len(MODEL_RELS),'fingerprint_sha256':bge_fingerprint_sha,'model_loads':0}})
if sha(OUT/'restricted_query_freeze.json') != '89f19fee34db2172cead05dbd25a565a8d48f90c96d7d5cc817a471597d34876': raise RuntimeError('restricted query freeze changed')
if sha(OUT/'query_embedding_manifest.json') != 'edef5f46b4e4b891255fe8e0ff21d18d00a7a8fdb91830e7ace6c78880d70351': raise RuntimeError('query cache audit changed')
if sha(OUT/'document_embedding_audit.json') != '264ae29777dd4ed522bf9cdfa65cd8f73a20f9873f00ba32fe67f11a1ae5994e': raise RuntimeError('document cache audit changed')
if not (query_manifest['transmit_items']==30 and query_manifest['transmit_tokens']==1726 and query_manifest['max_requests']==1): raise RuntimeError('query approval scope changed')

input_hashes_after={rel:sha(ROOT/rel) for rel in ALL_READ_ONLY_RELS}
if input_hashes_after != input_hashes_before: raise RuntimeError('read-only source/cache changed')
if any(query_manifest[k] for k in ['raw_cache_hits','normalized_cache_hits']): pass
if query_manifest['transmit_items'] > 64 or query_manifest['max_requests'] > 1: raise RuntimeError('approval cap unexpectedly exceeds one batch')
notebook_source_sha=source_only_notebook_sha(NOTEBOOK)
readme = f'''# 28 통합 holdout 후보 실행 전 봉인\n\n이 폴더는 결과를 보지 않은 CPU/offline preflight다. OLD와 STRUCT end-to-end 후보의 검색·reranker·evidence package 계약 및 실행 순서를 봉인했다. 공통 fused 작업목록은 Top50이며, OLD는 그 뒤 section/benefit 필터 후 첫 20개, STRUCT는 첫 20개를 사용한다. 현재 상태는 `sealed_awaiting_query_embedding_approval`이며 `authorized=false`다.\n\n- 질의: 30개 고유 문장, 기존 허용 캐시 raw hit {query_manifest['raw_cache_hits']}개, 전송 후보 {query_manifest['transmit_items']}개/{query_manifest['transmit_tokens']} cl100k tokens/최대 {query_manifest['max_requests']} request\n- 비용: 공식 단가를 이 offline run에서 확인하지 않아 미확정\n- 문서 embedding: OLD 327/327, STRUCT 147/147, float32 1536차원 finite\n- BGE: revision `{bge['revision']}`, local-only 계약만 검증; 모델 load/scoring 0\n- 실행: API/network/GPU/retrieval/BGE/LLM/new embedding 모두 0\n\n질의 평문은 권한을 0600으로 제한한 `restricted_query_freeze.json`에만 있다. 나머지 manifest/approval 파일에는 ID와 hash/count만 있다. 27 dataset seal과 SEALABLE reviewer provenance는 후보 preflight 준비 근거이며, 별도 사용자 승인 전 candidate 실행 권한을 뜻하지 않는다. gold·qualification은 ranking/BGE freeze 이후에만 읽도록 순서를 고정했다. 이전 수동 탐색 중 금지된 prior evaluation 경로의 파일 경로 하나가 출력된 사고는 contract/integrity provenance에 보존했으며, 내용 읽기·hash·후보 입력 사용은 없었다.\n'''
atomic_write_text(OUT/'README.md',readme)
output_names=['candidate_execution_contract.json','candidate_config_seal.json','restricted_query_freeze.json','query_embedding_manifest.json','query_embedding_approval.json','document_embedding_audit.json','README.md']
output_hashes={name:sha(OUT/name) for name in output_names}
integrity={'status':'PASS','candidate_core_sha256':candidate_core_sha,'query_approval_core_sha256':approval_sha,'notebook_source_only_sha256':notebook_source_sha,'record_count':30,'query_plaintext_public_manifest_count':0,'gold_master_qualification_reads':0,'query_approval_scope_unchanged':'30_items_1726_tokens_1_request','preserved_raw_sha256':{'restricted_query_freeze.json':'89f19fee34db2172cead05dbd25a565a8d48f90c96d7d5cc817a471597d34876','query_embedding_manifest.json':'edef5f46b4e4b891255fe8e0ff21d18d00a7a8fdb91830e7ace6c78880d70351','document_embedding_audit.json':'264ae29777dd4ed522bf9cdfa65cd8f73a20f9873f00ba32fe67f11a1ae5994e'},'manual_discovery_incident':candidate_contract['manual_discovery_incident'],'document_embedding_coverage':{'OLD':'327/327','STRUCT':'147/147'},'execution':{'fresh_cpu_kernel':True,'api':0,'network':0,'gpu':0,'model_load':0,'retrieval':0,'bge_scoring':0,'new_embedding':0,'llm':0},'input_hashes_before_after_exact':input_hashes_before==input_hashes_after,'output_hashes_excluding_integrity_and_manifest':output_hashes}
write_json('integrity.json',integrity)
manifest={'schema_version':'run_manifest_v1','provenance':'Codex coder agent','environment':'skn25','notebook_source_only_sha256':notebook_source_sha,'candidate_core_sha256':candidate_core_sha,'query_approval_core_sha256':approval_sha,'input_hashes':input_hashes_before,'output_hashes':{name:sha(OUT/name) for name in output_names+['integrity.json']},'self_hash_excluded':True,'execution_flags':integrity['execution']}
write_json('run_manifest.json',manifest)
# Recompute all non-self output hashes and final read-only hashes after the last artifact write.
if {rel:sha(ROOT/rel) for rel in ALL_READ_ONLY_RELS} != input_hashes_before: raise RuntimeError('final source/cache drift')
if json.loads((OUT/'query_embedding_manifest.json').read_text())['plaintext_included'] is not False: raise RuntimeError('public plaintext leak')
print(json.dumps({'status':'PASS','candidate_core_sha256':candidate_core_sha,'approval_core_sha256':approval_sha,'cache_hits':query_manifest['raw_cache_hits'],'transmit_items':query_manifest['transmit_items'],'tokens':query_manifest['transmit_tokens'],'max_requests':query_manifest['max_requests'],'OLD_embeddings':old_doc_hits,'STRUCT_embeddings':struct_doc_hits},ensure_ascii=False,indent=2))


{
  "status": "PASS",
  "candidate_core_sha256": "b6b00497bbccf306bfef86d7347dda6e965f5c8eefa08385d9609dbb6a3c6a43",
  "approval_core_sha256": "7ccf0f901b1ec4cd875e64978397a7b4529225c6c1ff9de84ba50b1cec58dfd3",
  "cache_hits": 0,
  "transmit_items": 30,
  "tokens": 1726,
  "max_requests": 1,
  "OLD_embeddings": 327,
  "STRUCT_embeddings": 147
}


## 승인 후 실행 단계

아래 세 단계는 source hash로 execution guard에 결속된다. 환경 플래그가 모두 0인 precheck에서는 외부 호출·GPU·gold load 없이 skip된다. query embedding만 승인된 외부 전송이며, retrieval/BGE는 local-only, scoring은 label-free freeze 검증 뒤에만 27번 restricted gold를 연다. LLM payload는 생성만 하고 호출하지 않는다.


In [2]:
# Freeze the exact approved core and the three stage sources before any API client or model import.
import atexit, csv, tempfile
def atomic_csv(path,rows,fieldnames):
    tmp=path.with_name('.'+path.name+'.tmp')
    with tmp.open('w',encoding='utf-8',newline='') as f: w=csv.DictWriter(f,fieldnames=fieldnames); w.writeheader(); w.writerows(rows)
    os.replace(tmp,path); return path
def atomic_npz(path,**arrays):
    tmp=path.with_name('.'+path.name+'.tmp')
    with tmp.open('wb') as f: np.savez_compressed(f,**arrays)
    os.replace(tmp,path); return path
def start_stage_ledger(name,details):
    state={'completed':False,'events':[{'event':'attempt','status':'started'}]}; path=OUT/(name+'_stage_ledger.json'); write_json(path.name,{'stage':name,'status':'started','events':state['events'],**details})
    def finalize_failure():
        if not state['completed']: state['events'].append({'event':'failure','status':'failed_or_interrupted'}); write_json(path.name,{'stage':name,'status':'failed_or_interrupted','events':state['events'],'failure_recorded_at_kernel_shutdown':True,**details})
    atexit.register(finalize_failure)
    def complete(extra=None):
        state['completed']=True; state['events'].append({'event':'completion','status':'completed'}); atexit.unregister(finalize_failure); write_json(path.name,{'stage':name,'status':'completed','events':state['events'],**details,**(extra or {})})
    def fail(reason):
        state['completed']=True; state['events'].append({'event':'failure','status':'failed','reason':reason}); atexit.unregister(finalize_failure); write_json(path.name,{'stage':name,'status':'failed','events':state['events'],'failure_reason':reason,**details})
    return complete,fail
stage_ids=['28-query-embedding-external','28-label-free-retrieval-bge','28-post-freeze-scoring-payload']
nb_doc=json.loads(NOTEBOOK.read_text(encoding='utf-8'))
def cell_source_sha(cell_id):
    cell=next((c for c in nb_doc['cells'] if c.get('id')==cell_id),None)
    if cell is None: raise RuntimeError('missing execution cell '+cell_id)
    source=''.join(cell.get('source',[])) if isinstance(cell.get('source',[]),list) else cell.get('source','')
    return text_sha(canon({'cell_id':cell_id,'source_text':source}))
stage_source_hashes={cell_id:cell_source_sha(cell_id) for cell_id in stage_ids}
approved_doc=json.loads((OUT/'query_embedding_approval.json').read_text(encoding='utf-8'))
if approved_doc['approval_core_sha256']!='7ccf0f901b1ec4cd875e64978397a7b4529225c6c1ff9de84ba50b1cec58dfd3': raise RuntimeError('approved core drift')
if approved_doc['approval_core']['candidate_core_sha256']!='b6b00497bbccf306bfef86d7347dda6e965f5c8eefa08385d9609dbb6a3c6a43': raise RuntimeError('candidate core drift')
candidate_seal_doc=json.loads((OUT/'candidate_config_seal.json').read_text()); candidate_seal_sha=sha(OUT/'candidate_config_seal.json')
guard_core={'schema_version':'candidate_execution_guard_v2','approval_core_sha256':approved_doc['approval_core_sha256'],'candidate_core_sha256':approved_doc['approval_core']['candidate_core_sha256'],'candidate_config_seal_sha256':candidate_seal_sha,'candidate_execution_input_hashes_sha256':text_sha(canon(candidate_seal_doc['execution_input_hashes'])),'bge_tree_fingerprint_sha256':candidate_seal_doc['bge_cache']['tree_fingerprint_sha256'],'dataset_seal_sha256':approved_doc['approval_core']['dataset_seal_sha256'],'dataset_manifest_sha256':candidate_seal_doc['source_hashes']['notebooks/data/27_integrated_holdout_dataset/draft_dataset_manifest.json'],'query_freeze_sha256':approved_doc['approval_core']['query_freeze_sha256'],'query_items':30,'query_tokens_cl100k':1726,'query_request_cap':1,'stage_source_hashes':stage_source_hashes,'approval_validity':'the user-approved query core remains valid because its candidate core and 30/1726/1 query-only transmission scope are unchanged; this new execution guard additionally binds revised local execution code and source/cache hashes','external_allowlist':['30 ordered query texts only'],'external_forbidden':approved_doc['approval_core']['transmitted_forbidden'],'local_stages':['OLD/STRUCT retrieval','local BGE','post-freeze gold scoring','LLM payload preflight only'],'llm_api_authorized':False}
guard_sha=text_sha(canon(guard_core))
write_json('execution_guard.json',{'guard_core':guard_core,'execution_guard_sha256':guard_sha,'status':'ready_for_main_controlled_execution'})
# Update preflight provenance without authorizing execution.
integrity_now=json.loads((OUT/'integrity.json').read_text()); integrity_now['execution_guard_sha256']=guard_sha; integrity_now['stage_source_hashes']=stage_source_hashes; write_json('integrity.json',integrity_now)
execution_command=f'''cd {ROOT}\nCUDA_VISIBLE_DEVICES=0 HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 TOKENIZERS_PARALLELISM=false RUN_APPROVED_28_QUERY_EMBEDDING=0 RUN_APPROVED_28_CACHE_RESUME=1 APPROVED_28_QUERY_EMBEDDING_SHA256={approved_doc['approval_core_sha256']} RUN_APPROVED_28_LOCAL_GPU=1 RUN_APPROVED_28_POST_FREEZE=1 APPROVED_28_EXECUTION_GUARD_SHA256={guard_sha} conda run -n skn25 jupyter nbconvert --to notebook --execute --inplace notebooks/28_integrated_holdout_candidate_evaluation.ipynb --ExecutePreprocessor.timeout=7200'''
readme_marker='\n## 승인 후 전체 fresh-kernel 순차 실행'; readme_now=(OUT/'README.md').read_text().split(readme_marker,1)[0]; readme_now += readme_marker+'\n\n```bash\n'+execution_command+'\n```\n\n이 API0 재개 명령은 이미 원자 저장된 query embedding cache를 승인 core·30개 hash·shape/dtype/finite·creation usage와 대조한 뒤 label-free retrieval/BGE freeze→post-freeze gold scoring 순으로 한 kernel에서 실행한다. query approval core의 candidate core와 30/1726/1 전송 범위는 그대로이며 새 execution guard만 재개 코드·전체 입력 hash에 추가 결속된다.\n'; atomic_write_text(OUT/'README.md',readme_now)
manifest_now=json.loads((OUT/'run_manifest.json').read_text()); manifest_now['execution_guard_sha256']=guard_sha; manifest_now['stage_source_hashes']=stage_source_hashes; manifest_now['output_hashes']['execution_guard.json']=sha(OUT/'execution_guard.json'); manifest_now['output_hashes']['integrity.json']=sha(OUT/'integrity.json'); manifest_now['output_hashes']['README.md']=sha(OUT/'README.md'); write_json('run_manifest.json',manifest_now)
print(json.dumps({'status':'CPU_PRECHECK_GUARD_READY','execution_guard_sha256':guard_sha,'stage_source_hashes':stage_source_hashes},indent=2))


{
  "status": "CPU_PRECHECK_GUARD_READY",
  "execution_guard_sha256": "7c898333022245daf594cb84e1f4dd4f5da3f161d016a1af5cd35f7e22772a29",
  "stage_source_hashes": {
    "28-query-embedding-external": "32d0ad2b0b5260b991afe14b359c84f0d792b54739a4afb6fe02bf0e2f373843",
    "28-label-free-retrieval-bge": "b2e94f291aafc0c55893e3737726f92b1595bf36fb6be147ef72fdc14722f741",
    "28-post-freeze-scoring-payload": "ec6a2237865ebbfb2bd53b33cf7c7a3dfee921c04888a5b9f79bd6317faa3381"
  }
}


In [3]:
# Exactly one approved query-only embedding request; a separately enabled API0 path recovers the complete atomic cache.
cache_resume=os.environ.get('RUN_APPROVED_28_QUERY_EMBEDDING','0')=='0' and os.environ.get('RUN_APPROVED_28_CACHE_RESUME','0')=='1'
if os.environ.get('RUN_APPROVED_28_QUERY_EMBEDDING','0')!='1' and not cache_resume:
    print('SKIP query embedding: RUN_APPROVED_28_QUERY_EMBEDDING=0')
else:
    def require_external(ok,message):
        if not ok: raise RuntimeError(message)
    guard_doc=json.loads((OUT/'execution_guard.json').read_text()); guard=guard_doc['guard_core']
    require_external(text_sha(canon(guard))==guard_doc['execution_guard_sha256']==os.environ.get('APPROVED_28_EXECUTION_GUARD_SHA256',''),'execution guard mismatch')
    require_external(cell_source_sha('28-query-embedding-external')==guard['stage_source_hashes']['28-query-embedding-external'],'external source changed')
    approval=json.loads((OUT/'query_embedding_approval.json').read_text()); core=approval['approval_core']
    require_external(text_sha(canon(core))==approval['approval_core_sha256']==os.environ.get('APPROVED_28_QUERY_EMBEDDING_SHA256',''),'approval mismatch')
    require_external(approval['approval_core_sha256']=='7ccf0f901b1ec4cd875e64978397a7b4529225c6c1ff9de84ba50b1cec58dfd3','unexpected approval')
    require_external(core['candidate_core_sha256']=='b6b00497bbccf306bfef86d7347dda6e965f5c8eefa08385d9609dbb6a3c6a43' and core['dataset_seal_sha256']==dataset_seal['dataset_seal_sha256'],'candidate/dataset mismatch')
    require_external(sha(OUT/'restricted_query_freeze.json')==core['query_freeze_sha256'],'query freeze mismatch')
    qfreeze=json.loads((OUT/'restricted_query_freeze.json').read_text())['ordered_queries']
    require_external(len(qfreeze)==core['item_cap']==30 and sum(q['cl100k_tokens'] for q in qfreeze)==core['token_cap_cl100k']==1726 and core['request_cap']==1,'approved caps mismatch')
    require_external([q['record_id'] for q in qfreeze]==core['ordered_miss_record_ids'] and [q['raw_text_sha256'] for q in qfreeze]==core['ordered_miss_raw_text_sha256'],'query order/hash mismatch')
    require_external(all(text_sha(q['query_text'])==q['raw_text_sha256'] for q in qfreeze),'query plaintext hash mismatch')
    complete_external,fail_external=start_stage_ledger('query_embedding',{'approval_core_sha256':approval['approval_core_sha256'],'execution_guard_sha256':guard_doc['execution_guard_sha256']})
    cache_dir=OUT/'embedding_cache/text-embedding-3-small'; cache_dir.mkdir(parents=True,exist_ok=True)
    cache_path=cache_dir/(approval['approval_core_sha256']+'.npz'); ledger_path=OUT/'query_embedding_attempt.json'
    require_external(not cache_resume or cache_path.is_file(),'API0 resume cache missing; external retry forbidden')
    if cache_path.exists():
        with np.load(cache_path,allow_pickle=False) as z:
            cached_vectors=z['embeddings'].copy(); cached_hashes=z['hashes'].tolist(); cached_approval=str(z['approval_core_sha256'].item()); creation_requests=int(z['creation_api_requests'].item()); creation_tokens=int(z['creation_input_tokens'].item())
        require_external(cached_vectors.shape==(30,1536) and cached_vectors.dtype==np.float32 and np.isfinite(cached_vectors).all(),'existing query cache invalid')
        require_external(cached_hashes==[q['raw_text_sha256'] for q in qfreeze] and cached_approval==approval['approval_core_sha256'],'existing query cache approval/order mismatch')
        require_external(creation_requests==1 and creation_tokens==1726,'existing query cache creation request/token mismatch')
        require_external(ledger_path.exists(),'cache without attempt ledger'); attempt=json.loads(ledger_path.read_text())
        require_external(attempt.get('approval_core_sha256')==approval['approval_core_sha256'] and attempt.get('requests_started')==1 and attempt.get('retries')==0 and attempt.get('status') in {'started','completed','completed_from_atomic_cache_recovery'},'cache attempt ledger mismatch')
        response_provenance={'provider_request_id':None,'provider_request_id_status':'not_available_on_CreateEmbeddingResponse','provider_model':'text-embedding-3-small','usage':{'input_tokens':creation_tokens},'data_count':30,'embedding_sha256':[sha_bytes(row.tobytes()) for row in cached_vectors],'provenance_source':'atomic_cache_saved_after_completed_provider_response','plaintext_included':False,'documents_or_gold_transmitted':False}
        write_json('query_embedding_response_provenance.json',response_provenance)
        attempt.update({'status':'completed_from_atomic_cache_recovery','requests_completed':1,'provider_model':'text-embedding-3-small','provider_request_id':None,'provider_request_id_status':'not_available_on_CreateEmbeddingResponse','usage_input_tokens':creation_tokens,'cache_path':str(cache_path.relative_to(ROOT)),'cache_sha256':sha(cache_path),'response_provenance_sha256':sha(OUT/'query_embedding_response_provenance.json'),'documents_or_gold_transmitted':False,'resume_api_requests':0}); write_json('query_embedding_attempt.json',attempt)
        write_json('query_embedding_usage.json',{'historical_approved_requests':1,'current_run_requests':0,'items':30,'input_tokens':creation_tokens,'model':'text-embedding-3-small','cache_sha256':sha(cache_path),'documents_or_gold_transmitted':False,'retries':0,'secret_persisted':False,'recovered_from_atomic_cache':True})
        complete_external({'requests_this_run':0,'historical_requests':1,'cache_sha256':sha(cache_path),'usage_input_tokens':creation_tokens})
        print('QUERY_CACHE_COMPLETE: API requests this run=0')
    else:
        require_external(not ledger_path.exists(),'attempt ledger already exists without complete cache; retry forbidden')
        attempt={'status':'started','approval_core_sha256':approval['approval_core_sha256'],'items':30,'token_cap_cl100k':1726,'request_cap':1,'requests_started':0,'retries':0,'documents_or_gold_transmitted':False}
        write_json('query_embedding_attempt.json',attempt)
        attempt['requests_started']=1; write_json('query_embedding_attempt.json',attempt)
        import time
        from openai import OpenAI
        started=time.perf_counter()
        try:
            response=OpenAI(max_retries=0).embeddings.create(model='text-embedding-3-small',input=[q['query_text'] for q in qfreeze])
        except Exception as exc:
            attempt.update({'status':'failed_no_retry','error_type':type(exc).__name__,'wall_seconds':time.perf_counter()-started}); write_json('query_embedding_attempt.json',attempt); fail_external(type(exc).__name__); raise
        vectors=np.asarray([row.embedding for row in response.data],dtype=np.float32)
        require_external(vectors.shape==(30,1536) and np.isfinite(vectors).all(),'embedding response shape/finite failure')
        atomic_npz(cache_path,embeddings=vectors,hashes=np.asarray([q['raw_text_sha256'] for q in qfreeze]),approval_core_sha256=np.asarray(approval['approval_core_sha256']),creation_api_requests=np.asarray(1,dtype=np.int64),creation_input_tokens=np.asarray(int(response.usage.prompt_tokens),dtype=np.int64))
        require_external(int(response.usage.prompt_tokens)<=1726,'provider usage exceeds approved cap')
        provider_request_id=getattr(response,'id',None); provider_request_id_sha256=text_sha(str(provider_request_id)) if provider_request_id is not None else None
        response_provenance={'provider_request_id_sha256':provider_request_id_sha256,'provider_request_id_status':'available' if provider_request_id is not None else 'not_available_on_CreateEmbeddingResponse','provider_model':response.model,'object':getattr(response,'object',None),'usage':response.usage.model_dump(mode='json'),'data_count':len(response.data),'data_indexes':[int(row.index) for row in response.data],'embedding_sha256':[sha_bytes(np.asarray(row.embedding,dtype=np.float32).tobytes()) for row in response.data],'plaintext_included':False,'documents_or_gold_transmitted':False}
        write_json('query_embedding_response_provenance.json',response_provenance)
        attempt.update({'status':'completed','requests_completed':1,'provider_model':response.model,'provider_request_id_sha256':provider_request_id_sha256,'provider_request_id_status':response_provenance['provider_request_id_status'],'usage_input_tokens':int(response.usage.prompt_tokens),'wall_seconds':time.perf_counter()-started,'cache_path':str(cache_path.relative_to(ROOT)),'cache_sha256':sha(cache_path),'response_provenance_sha256':sha(OUT/'query_embedding_response_provenance.json'),'documents_or_gold_transmitted':False}); write_json('query_embedding_attempt.json',attempt)
        write_json('query_embedding_usage.json',{'historical_approved_requests':1,'current_run_requests':1,'items':30,'input_tokens':int(response.usage.prompt_tokens),'model':response.model,'cache_sha256':sha(cache_path),'documents_or_gold_transmitted':False,'retries':0,'secret_persisted':False})
        complete_external({'requests_this_run':1,'cache_sha256':sha(cache_path),'usage_input_tokens':int(response.usage.prompt_tokens)})
        print(json.dumps({'status':'QUERY_EMBEDDING_COMPLETE','items':30,'input_tokens':int(response.usage.prompt_tokens),'requests':1,'cache_sha256':sha(cache_path)},indent=2))


QUERY_CACHE_COMPLETE: API requests this run=0


In [4]:
# Label-free OLD/STRUCT retrieval, evidence package construction, and local BGE freeze.
if os.environ.get('RUN_APPROVED_28_LOCAL_GPU','0')!='1':
    print('SKIP local retrieval/BGE: RUN_APPROVED_28_LOCAL_GPU=0')
else:
    import csv, re, time, resource
    from decimal import Decimal
    from collections import Counter, defaultdict
    guard_doc=json.loads((OUT/'execution_guard.json').read_text()); guard=guard_doc['guard_core']
    if text_sha(canon(guard))!=guard_doc['execution_guard_sha256'] or guard_doc['execution_guard_sha256']!=os.environ.get('APPROVED_28_EXECUTION_GUARD_SHA256',''): raise RuntimeError('execution guard mismatch before local stage')
    if cell_source_sha('28-label-free-retrieval-bge')!=guard['stage_source_hashes']['28-label-free-retrieval-bge']: raise RuntimeError('local stage source changed')
    complete_local,fail_local=start_stage_ledger('label_free_retrieval_bge',{'execution_guard_sha256':guard_doc['execution_guard_sha256']})
    def local_invalid(reason):
        fail_local(reason); write_json('terminal_invalid.json',{'stage':'label_free_retrieval_bge','status':'terminal_invalid','reason':reason}); raise RuntimeError(reason)
    candidate_seal_path=OUT/'candidate_config_seal.json'; candidate_seal_doc=json.loads(candidate_seal_path.read_text())
    if sha(candidate_seal_path)!=guard['candidate_config_seal_sha256'] or text_sha(canon(candidate_seal_doc['candidate_core']))!=candidate_seal_doc['candidate_core_sha256'] or candidate_seal_doc['candidate_core_sha256']!=guard['candidate_core_sha256']: local_invalid('candidate config seal mismatch')
    current_inputs={rel:sha(ROOT/rel) for rel in candidate_seal_doc['execution_input_hashes']}
    if current_inputs!=candidate_seal_doc['execution_input_hashes'] or text_sha(canon(current_inputs))!=guard['candidate_execution_input_hashes_sha256']: local_invalid('candidate-bound input raw hash mismatch')
    bge_root=ROOT/candidate_seal_doc['bge_cache']['root']; current_bge_files={str(p.relative_to(bge_root)):sha(p) for p in sorted(bge_root.rglob('*')) if p.is_file()}
    if current_bge_files!=candidate_seal_doc['bge_cache']['tree_files'] or text_sha(canon(current_bge_files))!=candidate_seal_doc['bge_cache']['tree_fingerprint_sha256'] or candidate_seal_doc['bge_cache']['tree_fingerprint_sha256']!=guard['bge_tree_fingerprint_sha256']: local_invalid('BGE cache tree fingerprint mismatch')
    if candidate_seal_doc['bge_cache']['custom_code_required'] or candidate_seal_doc['bge_cache']['custom_required_files'] or not all((ROOT/name).is_file() and sha(ROOT/name)==digest for name,digest in candidate_seal_doc['bge_cache']['required_model_files'].items()): local_invalid('BGE required-file/custom-code contract mismatch')
    if os.environ.get('CUDA_VISIBLE_DEVICES')!='0': raise RuntimeError('physical GPU0 must be the only visible device')
    approval=json.loads((OUT/'query_embedding_approval.json').read_text()); cache_path=OUT/'embedding_cache/text-embedding-3-small'/(approval['approval_core_sha256']+'.npz')
    if not cache_path.is_file(): raise RuntimeError('approved query cache missing')
    with np.load(cache_path,allow_pickle=False) as z:
        query_matrix=z['embeddings'].copy(); query_hashes=z['hashes'].tolist()
    if query_matrix.shape!=(30,1536) or query_matrix.dtype!=np.float32 or not np.isfinite(query_matrix).all(): raise RuntimeError('query cache invalid')
    qfreeze=json.loads((OUT/'restricted_query_freeze.json').read_text())['ordered_queries']
    if query_hashes!=[q['raw_text_sha256'] for q in qfreeze]: raise RuntimeError('query cache hash order')
    query_by_id={q['record_id']:q['query_text'] for q in qfreeze}; qvec={q['record_id']:query_matrix[i].astype(np.float64) for i,q in enumerate(qfreeze)}
    def vector_map(cache_rels):
        out={}
        for rel in cache_rels:
            with np.load(ROOT/rel,allow_pickle=False) as z:
                for h,v in zip(z['hashes'].tolist(),z['embeddings']): out.setdefault(str(h),v.astype(np.float64))
        return out
    old_vectors=vector_map(OLD_CACHE_RELS); struct_vectors=vector_map(STRUCT_CACHE_RELS)
    old_chunks=[json.loads(x) for x in (ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').read_text().splitlines() if x]
    struct_chunks=[json.loads(x) for x in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl').read_text().splitlines() if x]
    old_by={x['id']:x for x in old_chunks}; struct_by={x['chunk_id']:x for x in struct_chunks}
    old_doc_vec={x['id']:old_vectors[text_sha(x['document'])] for x in old_chunks}; struct_doc_vec={x['chunk_id']:struct_vectors[text_sha(x['retrieval_text'])] for x in struct_chunks}
    if len(old_doc_vec)!=327 or len(struct_doc_vec)!=147: raise RuntimeError('document vector coverage')
    RAW_TOKEN=re.compile(r'[0-9a-z가-힣]+')
    def normalized_text(value): return ' '.join(unicodedata.normalize('NFKC',str(value)).lower().split())
    def canonical_decimal(value):
        rendered=format(Decimal(str(value).replace(',','')).normalize(),'f'); rendered=rendered.rstrip('0').rstrip('.') if '.' in rendered else rendered; return '0' if rendered in {'','-0'} else rendered
    def search_tokens(value):
        text=normalized_text(value); tokens=list(RAW_TOKEN.findall(text))
        for run in re.findall(r'[가-힣](?:[가-힣 ]{0,38}[가-힣])?',text):
            joined=run.replace(' ','')
            for size in (2,3,4): tokens.extend(f'ko{size}_{joined[i:i+size]}' for i in range(max(0,len(joined)-size+1)))
        consumed=[]
        for m in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원',text): tokens.append(f'money_krw_{canonical_decimal(Decimal(m.group(1).replace(chr(44),"")) * 10000 + Decimal(m.group(2).replace(chr(44),"")) * 1000)}'); consumed.append(m.span())
        for m in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원',text):
            if any(a<=m.start() and m.end()<=b for a,b in consumed): continue
            tokens.append(f'money_krw_{canonical_decimal(Decimal(m.group(1).replace(chr(44),"")) * {"만":10000,"천":1000,None:1}[m.group(2)])}')
        for m in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*%',text): tokens.append(f'percent_{canonical_decimal(m.group(1))}')
        for m in re.finditer(r'(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)',text): tokens.append(f'period_{m.group(1) or "none"}_{canonical_decimal(m.group(2))}_{m.group(3)}')
        return tokens
    def bm25_rank(query,docs):
        q=search_tokens(query); tokenized={k:search_tokens(v) for k,v in docs.items()}; df=Counter(t for ts in tokenized.values() for t in set(ts)); avg=sum(map(len,tokenized.values()))/len(tokenized); scores={}
        for cid,tokens in tokenized.items():
            f=Counter(tokens); score=0.0
            for token in q:
                n=f[token]
                if n: score+=math.log(1+(len(tokenized)-df[token]+.5)/(df[token]+.5))*n*2.5/(n+1.5*(.25+.75*len(tokens)/avg))
            scores[cid]=score
        return sorted(scores,key=lambda x:(-scores[x],x)),scores
    def vector_rank(query_vector,vectors):
        distances={cid:float(np.sum((v-query_vector)**2,dtype=np.float64)) for cid,v in vectors.items()}
        if not all(math.isfinite(v) for v in distances.values()): raise RuntimeError('nonfinite vector distance')
        return sorted(distances,key=lambda x:(distances[x],x)),distances
    def fused_top50(bm25,vector):
        br={x:i for i,x in enumerate(bm25[:50],1)}; vr={x:i for i,x in enumerate(vector[:50],1)}; ids=set(br)|set(vr); scores={x:(.6/(60+br[x]) if x in br else 0)+(.4/(60+vr[x]) if x in vr else 0) for x in ids}; best={x:min(br.get(x,10**9),vr.get(x,10**9)) for x in ids}; ranked=sorted(ids,key=lambda x:(-scores[x],best[x],x))[:50]; return ranked,br,vr,scores
    rankings=[]; old_candidates={}; struct_candidates={}
    old_docs={x['id']:x['document'] for x in old_chunks}; struct_docs={x['chunk_id']:x['retrieval_text'] for x in struct_chunks}
    for rid in [q['record_id'] for q in qfreeze]:
        ob,_=bm25_rank(query_by_id[rid],old_docs); ov,_=vector_rank(qvec[rid],old_doc_vec); of,obr,ovr,ofs=fused_top50(ob,ov); leaf=[x for x in of if old_by[x]['metadata']['level'] in {'section','benefit'}][:20]
        sb,_=bm25_rank(query_by_id[rid],struct_docs); sv,_=vector_rank(qvec[rid],struct_doc_vec); sf,sbr,svr,sfs=fused_top50(sb,sv); sd20=sf[:20]
        if len(of)!=50 or len(sf)!=50 or len(leaf)!=20 or len(sd20)!=20: raise RuntimeError('ranking depth contract')
        old_candidates[rid]=leaf; struct_candidates[rid]=sd20
        for corpus,ranked,br,vr,fs in [('OLD',of,obr,ovr,ofs),('STRUCT',sf,sbr,svr,sfs)]:
            rankings.append({'record_id':rid,'corpus':corpus,'top50':[{'rank':i,'chunk_id':cid,'bm25_rank':br.get(cid),'vector_rank':vr.get(cid),'rrf_score':fs[cid]} for i,cid in enumerate(ranked,1)],'d20':old_candidates[rid] if corpus=='OLD' else struct_candidates[rid]})
    ranking_path=OUT/'label_free_rankings.jsonl'; atomic_write_text(ranking_path,''.join(canon(x)+'\n' for x in rankings))
    # Frozen query-only classifier.
    proper_rules=(('proper_issuer_product',re.compile(r'(?:어느|어떤)\s*(?:(?:카드사|은행|회사)\s*)?상품(?:인가)?$')),('proper_issuer_direct',re.compile(r'(?:어느\s*)?(?:카드사|은행|회사)(?:인가)?$')),('proper_issuer_noun',re.compile(r'(?:발급사|발행사)(?:는|은|가|인가)?$')),('proper_where_action',re.compile(r'어디서\s*(?:발급|발행|출시)')))
    numeric_rules=(('numeric_direct_amount',re.compile(r'얼마(?:인가|나)?$')),('numeric_direct_count',re.compile(r'몇\s*(?:원|%|퍼센트|마일|마일리지|포인트|회|개월|일|년)(?:인가)?$')),('numeric_direct_how_much',re.compile(r'얼마나\s*(?:할인|적립|차감|청구)')),('numeric_target_end',re.compile(r'(?:할인율|적립률|연회비|수수료|(?:할인|적립)\s*(?:금액|한도)|(?:월|연간)\s*(?:할인|적립)?\s*한도|리터당\s*할인\s*금액|(?:마일리지|포인트)\s*적립\s*기준|실적\s*(?:금액|기준)|이용\s*(?:횟수|기간))(?:은|는|이|가|인가)?$')))
    def classify(text):
        t=' '.join(unicodedata.normalize('NFKC',text).lower().split()).rstrip(' ?.!,。？！')
        for rule,p in proper_rules:
            if p.search(t): return 'proper_noun'
        if '연회비 면제 조건' not in t:
            for rule,p in numeric_rules:
                if p.search(t): return 'numeric_condition'
        return 'semantic'
    classifications={rid:classify(query_by_id[rid]) for rid in query_by_id}
    # Exact structural one-hop bundle builder.
    hierarchy=[json.loads(x) for x in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl').read_text().splitlines() if x]; node_by={x['node_id']:x for x in hierarchy}; children=defaultdict(list)
    for n in hierarchy:
        if n['parent_id'] is not None: children[n['parent_id']].append(n)
    for rows in children.values(): rows.sort(key=lambda n:((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
    bundles=[]
    for rid,d20 in struct_candidates.items():
        by_card=defaultdict(list)
        for rank,cid in enumerate(d20,1): by_card[struct_by[cid]['metadata']['card_key']].append((rank,cid))
        for card,seeds in sorted(by_card.items(),key=lambda x:(x[1][0][0],x[0])):
            best_rank,seed_id=seeds[0]; seed=struct_by[seed_id]; node=node_by[seed['metadata']['node_id']]; selected=[]; trace=[]; parent_context=None
            def add(cid,relation,source_node):
                if len(selected)>=5 or cid in selected: return
                if struct_by[cid]['metadata']['card_key']!=card: raise RuntimeError('cross-card bundle')
                selected.append(cid); trace.append({'order':len(selected),'chunk_id':cid,'relation':relation,'source_node_id':source_node})
            add(seed_id,'best_rrf_seed',node['node_id'])
            same=[struct_by[c] for c in node['search_chunk_ids'] if c!=seed_id]; same.sort(key=lambda c:(abs(c['metadata']['part_index']-seed['metadata']['part_index']),c['metadata']['part_index'],c['chunk_id']))
            for c in same: add(c['chunk_id'],'same_node_adjacent_part',node['node_id'])
            parent=node_by.get(node['parent_id'])
            if parent is not None and parent['parent_id'] is not None:
                if not parent['heading_only']:
                    for cid in parent['search_chunk_ids']: add(cid,'non_root_immediate_parent_direct_body',parent['node_id'])
                else:
                    parent_context=parent['heading_text']; seed_line=node['heading_line_number'] if node['heading_line_number'] is not None else 10**12; siblings=[n for n in children[parent['node_id']] if not n['heading_only'] and n['search_chunk_ids']]; siblings.sort(key=lambda n:(abs((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12)-seed_line),(n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
                    for n in siblings:
                        for cid in n['search_chunk_ids']: add(cid,'heading_only_parent_same_parent_direct_body_child',parent['node_id'])
            if node['parent_id'] is not None:  # Root seed itself is allowed; root-child fanout is forbidden.
                for child in children[node['node_id']]:
                    if not child['heading_only']:
                        for cid in child['search_chunk_ids']: add(cid,'seed_node_direct_child',node['node_id'])
            for _,cid in seeds[1:]: add(cid,'same_card_other_top20_seed',node['node_id'])
            sections=['[카드]\n'+seed['metadata']['issuer']+' > '+seed['metadata']['card_name']]
            if parent_context: sections.append('[상위 제목]\n'+parent_context)
            for i,cid in enumerate(selected,1):
                c=struct_by[cid]; heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)'; sections.append(f'[근거 {i} 경로]\n{heading}\n[근거 {i} 본문]\n{c["evidence_text"]}')
            text='\n\n'.join(sections); bundles.append({'record_id':rid,'card_key':card,'best_seed_rrf_rank':best_rank,'best_seed_chunk_id':seed_id,'selected_chunk_ids':selected,'bundle_text':text,'bundle_sha256':text_sha(text),'relation_trace':trace})
    if len({(x['record_id'],x['card_key']) for x in bundles})!=len(bundles): raise RuntimeError('duplicate structural bundle')
    # OLD safe title augmentation and pair list; no labels are loaded.
    heading_re=re.compile(r'^#{1,6}\s+(.+?)\s*$',re.MULTILINE)
    def old_aug(cid):
        c=old_by[cid]; m=heading_re.search(c['document']); path=[c['metadata']['issuer'],c['metadata']['card_name'],c['metadata']['level']]+([m.group(1).strip()] if m else []); return '[문서 경로]\n'+' > '.join(path)+'\n\n[본문]\n'+c['document']
    pairs=[]
    for rid,d20 in old_candidates.items():
        if classifications[rid]=='semantic':
            for rank,cid in enumerate(d20,1): pairs.append({'pair_key':'OLD|'+rid+'|'+cid,'system':'OLD','record_id':rid,'candidate_id':cid,'original_rank':rank,'query_text':query_by_id[rid],'document_text':old_aug(cid)})
    for b in bundles: pairs.append({'pair_key':'STRUCT|'+b['record_id']+'|'+b['card_key'],'system':'STRUCT','record_id':b['record_id'],'candidate_id':b['card_key'],'original_rank':b['best_seed_rrf_rank'],'query_text':query_by_id[b['record_id']],'document_text':b['bundle_text']})
    if len({p['pair_key'] for p in pairs})!=len(pairs): raise RuntimeError('duplicate BGE pair')
    import gc, torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    model_path=ROOT/'.cache/reranker/bge-reranker-v2-m3'; audit_tokenizer=AutoTokenizer.from_pretrained(model_path,local_files_only=True,trust_remote_code=False)
    for b in bundles:
        if len(audit_tokenizer.encode(b['bundle_text'],add_special_tokens=False))>4096: local_invalid('STRUCT bundle token overflow package failure')
    del audit_tokenizer; gc.collect()
    torch.cuda.set_device(0); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)
    def score_attempt(batch_size):
        tokenizer_local=model_local=None; values=[]; load_seconds_local=0.0; score_started=None
        try:
            tokenizer_local=AutoTokenizer.from_pretrained(model_path,local_files_only=True,trust_remote_code=False)
            load_started=time.perf_counter(); model_local=AutoModelForSequenceClassification.from_pretrained(model_path,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0'); load_seconds_local=time.perf_counter()-load_started; score_started=time.perf_counter()
            with torch.inference_mode():
                for start in range(0,len(pairs),batch_size):
                    batch=pairs[start:start+batch_size]; encoded=outputs=logits_tensor=None
                    try:
                        encoded=tokenizer_local([p['query_text'] for p in batch],[p['document_text'] for p in batch],padding=True,truncation='only_second',max_length=8192,return_tensors='pt'); encoded={k:v.to('cuda:0') for k,v in encoded.items()}; outputs=model_local(**encoded); logits_tensor=outputs.logits.float().reshape(-1).cpu(); values.extend(logits_tensor.numpy().tolist())
                    finally:
                        del logits_tensor,outputs,encoded
            torch.cuda.synchronize(0); return {'status':'ok','values':values,'load_seconds':load_seconds_local,'score_seconds':time.perf_counter()-score_started}
        except RuntimeError as exc:
            is_oom=isinstance(exc,torch.cuda.OutOfMemoryError) or 'out of memory' in str(exc).lower(); error_type=type(exc).__name__; del exc
            if is_oom: return {'status':'oom','error_type':error_type}
            raise
        finally:
            del model_local,tokenizer_local,values; gc.collect()
            try: torch.cuda.synchronize(0)
            except Exception: pass
            torch.cuda.empty_cache()
    attempt_result=score_attempt(2); oom=attempt_result['status']=='oom'; final_batch=2
    if oom:
        del attempt_result; gc.collect(); torch.cuda.synchronize(0); torch.cuda.empty_cache(); final_batch=1; attempt_result=score_attempt(1)
        if attempt_result['status']=='oom': local_invalid('batch1 OOM terminal after discarded batch2 partial scores')
    logits=attempt_result.pop('values'); load_seconds=attempt_result['load_seconds']; score_seconds=attempt_result['score_seconds']; del attempt_result; gc.collect(); torch.cuda.synchronize(0); torch.cuda.empty_cache()
    if len(logits)!=len(pairs) or not np.isfinite(np.asarray(logits)).all(): raise RuntimeError('BGE score count/finite')
    scores={p['pair_key']:float(v) for p,v in zip(pairs,logits)}
    pair_public=[{k:v for k,v in p.items() if k not in {'query_text','document_text'}}|{'raw_logit':scores[p['pair_key']],'query_text_sha256':text_sha(p['query_text']),'document_text_sha256':text_sha(p['document_text'])} for p in pairs]
    pair_path=OUT/'bge_pair_scores.jsonl'; atomic_write_text(pair_path,''.join(canon(x)+'\n' for x in pair_public))
    bundle_by={(b['record_id'],b['card_key']):b for b in bundles}; final_rankings=[]; packages=[]
    for rid in query_by_id:
        od20=old_candidates[rid]
        if classifications[rid]=='semantic': od20=sorted(od20,key=lambda cid:(-scores['OLD|'+rid+'|'+cid],old_candidates[rid].index(cid)+1,cid))
        seen=[]
        for cid in od20:
            card=old_by[cid]['metadata']['card_key']
            if card not in seen: seen.append(card)
        old_cards=seen[:3]; final_rankings.append({'record_id':rid,'system':'OLD','route':'reranker' if classifications[rid]=='semantic' else 'no_reranker','top3_cards':old_cards,'ranked_candidate_ids':od20})
        for card_rank,card in enumerate(old_cards,1):
            ids=[cid for cid in od20 if old_by[cid]['metadata']['card_key']==card][:5]; packages.append({'record_id':rid,'system':'OLD','card_rank':card_rank,'card_key':card,'selected_chunk_ids':ids,'evidence':[{'evidence_id':f'OLD-{card_rank}-{i}','issuer':old_by[cid]['metadata']['issuer'],'card_name':old_by[cid]['metadata']['card_name'],'heading':old_by[cid]['metadata'].get('section',''),'text':old_by[cid]['document']} for i,cid in enumerate(ids,1)]})
        sb=[b for b in bundles if b['record_id']==rid]; sb.sort(key=lambda b:(-scores['STRUCT|'+rid+'|'+b['card_key']],b['best_seed_rrf_rank'],b['card_key'])); struct_cards=[b['card_key'] for b in sb[:3]]; final_rankings.append({'record_id':rid,'system':'STRUCT','route':'reranker','top3_cards':struct_cards,'ranked_candidate_ids':[b['card_key'] for b in sb]})
        for card_rank,b in enumerate(sb[:3],1): packages.append({'record_id':rid,'system':'STRUCT','card_rank':card_rank,'card_key':b['card_key'],'selected_chunk_ids':b['selected_chunk_ids'],'evidence':[{'evidence_id':f'STRUCT-{card_rank}-{i}','issuer':struct_by[cid]['metadata']['issuer'],'card_name':struct_by[cid]['metadata']['card_name'],'heading':' > '.join(struct_by[cid]['heading_path']),'text':struct_by[cid]['evidence_text']} for i,cid in enumerate(b['selected_chunk_ids'],1)]})
    ranking_final_path=OUT/'candidate_rankings_freeze.jsonl'; atomic_write_text(ranking_final_path,''.join(canon(x)+'\n' for x in final_rankings))
    package_path=OUT/'evidence_packages_freeze.jsonl'; atomic_write_text(package_path,''.join(canon(x)+'\n' for x in packages))
    bundle_path=OUT/'structural_bundles_freeze.jsonl'; atomic_write_text(bundle_path,''.join(canon({k:v for k,v in b.items() if k!='bundle_text'})+'\n' for b in bundles))
    resources={'pair_count':len(pairs),'finite_pairs':len(logits),'final_batch_size':final_batch,'oom_batch2':oom,'load_seconds':load_seconds,'scoring_seconds':score_seconds,'pairs_per_second':len(pairs)/score_seconds,'peak_allocated_gib':torch.cuda.max_memory_allocated(0)/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved(0)/2**30,'dtype':'float16','max_length':8192,'truncation':'only_second','physical_gpu':0,'api_requests':0,'network':0,'new_embeddings':0,'model_revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'}; write_json('local_bge_resources.json',resources)
    freeze_core={'schema_version':'label_free_execution_freeze_v1','candidate_core_sha256':guard['candidate_core_sha256'],'query_cache_sha256':sha(cache_path),'ranking_worklist_sha256':sha(ranking_path),'pair_scores_sha256':sha(pair_path),'candidate_rankings_sha256':sha(ranking_final_path),'evidence_packages_sha256':sha(package_path),'structural_bundles_sha256':sha(bundle_path),'resources_sha256':sha(OUT/'local_bge_resources.json'),'local_stage_source_sha256':cell_source_sha('28-label-free-retrieval-bge'),'gold_master_qualification_scoring_reads_before_freeze':0,'label_free':True}
    freeze_sha=text_sha(canon(freeze_core)); write_json('label_free_execution_freeze.json',{'freeze_core':freeze_core,'freeze_sha256':freeze_sha,'status':'sealed_before_gold_load'})
    complete_local({'freeze_sha256':freeze_sha,'pair_count':len(pairs),'final_batch_size':final_batch,'oom_batch2':oom})
    print(json.dumps({'status':'LABEL_FREE_FREEZE_COMPLETE','freeze_sha256':freeze_sha,'pairs':len(pairs),'packages':len(packages),'oom_batch2':oom},indent=2))


SKIP local retrieval/BGE: RUN_APPROVED_28_LOCAL_GPU=0


In [5]:
# Restricted gold/scoring is opened only after the label-free freeze is validated. No LLM call occurs here.
if os.environ.get('RUN_APPROVED_28_POST_FREEZE','0')!='1':
    print('SKIP post-freeze scoring: RUN_APPROVED_28_POST_FREEZE=0')
else:
    import csv, tiktoken
    from collections import Counter
    guard_doc=json.loads((OUT/'execution_guard.json').read_text()); guard=guard_doc['guard_core']
    if text_sha(canon(guard))!=guard_doc['execution_guard_sha256'] or guard_doc['execution_guard_sha256']!=os.environ.get('APPROVED_28_EXECUTION_GUARD_SHA256',''): raise RuntimeError('post-freeze guard mismatch')
    if cell_source_sha('28-post-freeze-scoring-payload')!=guard['stage_source_hashes']['28-post-freeze-scoring-payload']: raise RuntimeError('post-freeze source changed')
    freeze_doc=json.loads((OUT/'label_free_execution_freeze.json').read_text()); freeze=freeze_doc['freeze_core']
    if text_sha(canon(freeze))!=freeze_doc['freeze_sha256'] or freeze['local_stage_source_sha256']!=guard['stage_source_hashes']['28-label-free-retrieval-bge']: raise RuntimeError('label-free freeze mismatch')
    for name,key in [('label_free_rankings.jsonl','ranking_worklist_sha256'),('bge_pair_scores.jsonl','pair_scores_sha256'),('candidate_rankings_freeze.jsonl','candidate_rankings_sha256'),('evidence_packages_freeze.jsonl','evidence_packages_sha256'),('structural_bundles_freeze.jsonl','structural_bundles_sha256'),('local_bge_resources.json','resources_sha256')]:
        if sha(OUT/name)!=freeze[key]: raise RuntimeError('label-free artifact drift '+name)
    complete_post,fail_post=start_stage_ledger('post_freeze_scoring_payload',{'execution_guard_sha256':guard_doc['execution_guard_sha256'],'freeze_sha256':freeze_doc['freeze_sha256']})
    def post_invalid(reason):
        fail_post(reason); write_json('terminal_invalid.json',{'stage':'post_freeze_scoring_payload','status':'terminal_invalid','reason':reason}); raise RuntimeError(reason)
    # The following paths are intentionally first referenced only after all checks above.
    D27=ROOT/'notebooks/data/27_integrated_holdout_dataset'
    candidate_seal_doc=json.loads((OUT/'candidate_config_seal.json').read_text()); dataset_manifest_path=D27/'draft_dataset_manifest.json'
    if sha(dataset_manifest_path)!=guard['dataset_manifest_sha256'] or sha(dataset_manifest_path)!=candidate_seal_doc['source_hashes'][str(dataset_manifest_path.relative_to(ROOT))]: post_invalid('dataset manifest hash mismatch before qualification load')
    dataset_manifest=json.loads(dataset_manifest_path.read_text())
    gold_path=D27/'master_gold.jsonl'; qual_path=D27/'qualification_by_card.jsonl'; scoring_path=D27/'draft_scoring_contract.json'; projection_path=D27/'candidate_projection.jsonl'; claims_path=D27/'atomic_claims.json'; sources_path=D27/'source_components.jsonl'
    if sha(qual_path)!=dataset_manifest['artifacts']['qualification_by_card.jsonl']: post_invalid('qualification_by_card hash mismatch before post-freeze scoring')
    expected_hashes=dataset_seal['seal_core']['hashes']
    if sha(gold_path)!=expected_hashes['gold'] or sha(scoring_path)!=expected_hashes['scoring'] or sha(projection_path)!=expected_hashes['projection'] or sha(claims_path)!=expected_hashes['claims'] or sha(sources_path)!=expected_hashes['source_components']: raise RuntimeError('sealed scoring input hash mismatch')
    scoring=json.loads(scoring_path.read_text()); master=[json.loads(x) for x in gold_path.read_text().splitlines() if x]; qualification=[json.loads(x) for x in qual_path.read_text().splitlines() if x]; projections=[json.loads(x) for x in projection_path.read_text().splitlines() if x]; claims=json.loads(claims_path.read_text())['claims']
    qfreeze=json.loads((OUT/'restricted_query_freeze.json').read_text())['ordered_queries']; query_by_id={q['record_id']:q['query_text'] for q in qfreeze}
    old_chunks=[json.loads(x) for x in (ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').read_text().splitlines() if x]; struct_chunks=[json.loads(x) for x in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl').read_text().splitlines() if x]; old_by={x['id']:x for x in old_chunks}; struct_by={x['chunk_id']:x for x in struct_chunks}
    if scoring['schema_version']!='integrated_holdout_scoring_v6' or len(master)!=30 or len(qualification)!=300: raise RuntimeError('scoring schema/count mismatch')
    if [r['record_id'] for r in master]!=dataset_seal['seal_core']['record_ids']: raise RuntimeError('gold record order mismatch')
    active={c['atomic_claim_id']:c for c in claims if c.get('evaluation_status')!='retired_non_evaluation'}; proj={(p['atomic_claim_id'],p['corpus']):p for p in projections if p['projection_scope']=='atomic_claim'}
    packages=[json.loads(x) for x in (OUT/'evidence_packages_freeze.jsonl').read_text().splitlines() if x]; rankings=[json.loads(x) for x in (OUT/'candidate_rankings_freeze.jsonl').read_text().splitlines() if x]; worklists=[json.loads(x) for x in (OUT/'label_free_rankings.jsonl').read_text().splitlines() if x]
    package_by={(p['record_id'],p['system'],p['card_key']):p for p in packages}; ranking_by={(r['record_id'],r['system']):r for r in rankings}; work_by={(r['record_id'],r['corpus']):r for r in worklists}
    if len(ranking_by)!=60 or len(work_by)!=60: raise RuntimeError('ranking rows')
    CARD_ID_BY_PREFIX={'BC':'BC','NH':'NH','hana':'HANA','hyundai':'HYUNDAI','ibk':'IBK','kookmin':'KB','lotte':'LOTTE','samsung':'SAMSUNG','shinhan':'SHINHAN','woori':'WOORI'}
    def card_id(card_key):
        prefix=card_key.split('/',1)[0]
        if prefix not in CARD_ID_BY_PREFIX: raise RuntimeError('unknown card_key prefix '+prefix)
        return CARD_ID_BY_PREFIX[prefix]
    def claim_supported(record_id,system,card_key,claim_id,selected_ids):
        p=proj[(claim_id,system)]; mapping=p['searchable_eligible_projection']['component_chunk_ids']; selected=set(selected_ids); return all(bool(selected & set(ids)) for ids in mapping.values())
    def graph_supported(record,system,card_key):
        cid=card_id(card_key); graph=record['qualification_graphs'].get(cid); pkg=package_by.get((record['record_id'],system,card_key))
        if graph is None or pkg is None: return False
        supported={claim_id:claim_supported(record['record_id'],system,card_key,claim_id,pkg['selected_chunk_ids']) for branch in ('all_of','any_of') for claim_id,*_ in graph.get(branch,[])}
        return all(supported[x[0]] for x in graph.get('all_of',[])) and (not graph.get('any_of') or any(supported[x[0]] for x in graph['any_of']))
    rows=[]
    for record in master:
        rid=record['record_id']; expected=list(record['expected_card_ids'])
        for system in ('OLD','STRUCT'):
            rank=ranking_by[(rid,system)]; returned_keys=rank['top3_cards']; returned=[card_id(x) for x in returned_keys]
            if len(returned)!=len(set(returned)) or len(returned)>3: raise RuntimeError('returned card contract')
            d20=work_by[(rid,system)]['d20']; source_by=old_by if system=='OLD' else struct_by; candidate_cards={card_id(source_by[x]['metadata']['card_key']) for x in d20}
            supported_cards={card_id(k) for k in returned_keys if graph_supported(record,system,k)}
            supported_claims=set(); supported_units=total_units=0
            for expected_id in expected:
                graph=record['qualification_graphs'][expected_id]; returned_key=next((k for k in returned_keys if card_id(k)==expected_id),None); selected_ids=package_by[(rid,system,returned_key)]['selected_chunk_ids'] if returned_key is not None else []
                for claim_id,*_ in graph.get('all_of',[]):
                    total_units+=1
                    if returned_key is not None and claim_supported(rid,system,returned_key,claim_id,selected_ids): supported_units+=1; supported_claims.add(claim_id)
                any_of=graph.get('any_of',[])
                if any_of:
                    total_units+=1; branch_supported=[]
                    for claim_id,*_ in any_of:
                        ok=returned_key is not None and claim_supported(rid,system,returned_key,claim_id,selected_ids); branch_supported.append(ok)
                        if ok: supported_claims.add(claim_id)
                    supported_units+=int(any(branch_supported))
            if expected:
                if total_units<=0: raise RuntimeError('positive record has empty required graph units')
                hits=len(set(returned)&set(expected)); precision=hits/len(returned) if returned else 0.0; recall=hits/len(expected); supported_recall=len(set(expected)&set(returned)&supported_cards)/len(expected); claim_coverage=supported_units/total_units; negative_correct=None; e2e=int(set(returned)==set(expected) and supported_recall==1 and claim_coverage==1)
            else:
                precision=recall=supported_recall=claim_coverage=None; negative_correct=int(not any(q['qualified'] for q in qualification if q['record_id']==rid and q['card_id'] in returned)); e2e=negative_correct
            ceiling=len(set(expected)&candidate_cards)/len(expected) if expected else None
            rows.append({'record_id':rid,'system':system,'positive':bool(expected),'returned_card_ids':returned,'expected_card_count':len(expected),'candidate_card_recall_d20':ceiling,'card_precision_at_3':precision,'card_recall_at_3':recall,'supported_card_recall_at_3':supported_recall,'required_claim_coverage':claim_coverage,'negative_correctness':negative_correct,'retrieval_end_to_end_exact_success':e2e,'supported_claim_ids':sorted(supported_claims)})
    if len(rows)!=60 or any(not (0<=v<=1) for r in rows for v in [r['card_precision_at_3'],r['card_recall_at_3'],r['supported_card_recall_at_3'],r['required_claim_coverage'],r['negative_correctness'],r['candidate_card_recall_d20']] if v is not None): raise RuntimeError('metric rows/range')
    metrics_path=OUT/'retrieval_metrics.csv'; fields=list(rows[0]); atomic_csv(metrics_path,[{**r,'returned_card_ids':canon(r['returned_card_ids']),'supported_claim_ids':canon(r['supported_claim_ids'])} for r in rows],fields)
    summary=[]
    for system in ('OLD','STRUCT'):
        sr=[r for r in rows if r['system']==system]; pos=[r for r in sr if r['positive']]; neg=[r for r in sr if not r['positive']]
        summary.append({'system':system,'positive_records':26,'negative_records':4,'card_precision_at_3':sum(r['card_precision_at_3'] for r in pos)/26,'card_recall_at_3':sum(r['card_recall_at_3'] for r in pos)/26,'supported_card_recall_at_3':sum(r['supported_card_recall_at_3'] for r in pos)/26,'required_claim_coverage':sum(r['required_claim_coverage'] for r in pos)/26,'candidate_card_recall_d20':sum(r['candidate_card_recall_d20'] for r in pos)/26,'negative_correctness_count':sum(r['negative_correctness'] for r in neg),'retrieval_e2e_count':sum(r['retrieval_end_to_end_exact_success'] for r in sr)})
    write_json('retrieval_summary.json',{'schema_version':'retrieval_scoring_v6_projection_v1','scoring_contract_sha256':expected_hashes['scoring'],'systems':summary,'note':'retrieval evidence projection only; not LLM answer scoring'})
    paired=[]; wlt=Counter()
    for rid in dataset_seal['seal_core']['record_ids']:
        a=next(r for r in rows if r['record_id']==rid and r['system']=='OLD'); b=next(r for r in rows if r['record_id']==rid and r['system']=='STRUCT'); metric='supported_card_recall_at_3' if a['positive'] else 'negative_correctness'; delta=b[metric]-a[metric]; outcome='win' if delta>1e-12 else ('loss' if delta< -1e-12 else 'tie'); wlt[outcome]+=1; paired.append({'record_id':rid,'metric':metric,'old':a[metric],'struct':b[metric],'delta_struct_minus_old':delta,'outcome':outcome})
    atomic_csv(OUT/'retrieval_paired.csv',paired,list(paired[0]))
    write_json('retrieval_wlt.json',dict(wlt))
    summary_by={x['system']:x for x in summary}; absolute=scoring['selection_contract']['absolute']; critical_ids=set(scoring['selection_contract']['critical_cohort_record_ids']); critical_retrieval_regressions=[]
    for rid in critical_ids:
        old_row=next(x for x in rows if x['record_id']==rid and x['system']=='OLD'); struct_row=next(x for x in rows if x['record_id']==rid and x['system']=='STRUCT')
        if old_row['retrieval_end_to_end_exact_success']==1 and struct_row['retrieval_end_to_end_exact_success']==0: critical_retrieval_regressions.append(rid)
    retrieval_gates={system:{'supported_card_recall_at_3':summary_by[system]['supported_card_recall_at_3']>=absolute['supported_card_recall_at_3_macro_min'],'required_claim_coverage':summary_by[system]['required_claim_coverage']>=absolute['required_claim_coverage_macro_min'],'negative_correctness_4_of_4':summary_by[system]['negative_correctness_count']==4} for system in ('OLD','STRUCT')}
    write_json('retrieval_decision.json',{'status':'awaiting_llm_generation_no_retrieval_winner','retrieval_only_gates':retrieval_gates,'critical_retrieval_regression_record_ids_struct_vs_old':sorted(critical_retrieval_regressions),'negative_gate_evaluable':True,'llm_answer_gates_pending':['end_to_end_exact_success_count_min_21_of_30','positive_citation_ownership_1.0','all_derived_error_totals_0'],'selection_eligible':False,'reason':'sealed scoring contract selects on end-to-end answer quality; retrieval-only results cannot choose a winner'})
    # LLM payloads use only query and frozen anonymous evidence; gold/scoring values are never inserted.
    system_prompt='''제공된 카드 근거만 사용해 질문에 답하세요. 카드·발급사·혜택·숫자·단위·조건 주장은 실제 evidence_id를 인용해야 합니다. 근거가 부족하면 insufficient_evidence를 true로 두고 부족함을 설명하세요. 발급 가능성, 시장 전체 최적, 개인 자격은 주장하지 마세요.'''
    schema={'type':'object','additionalProperties':False,'required':['summary','cards','insufficient_evidence'],'properties':{'summary':{'type':'string'},'cards':{'type':'array','maxItems':3,'items':{'type':'object','additionalProperties':False,'required':['issuer','card_name','claims'],'properties':{'issuer':{'type':'string'},'card_name':{'type':'string'},'claims':{'type':'array','items':{'type':'object','additionalProperties':False,'required':['text','citations'],'properties':{'text':{'type':'string'},'citations':{'type':'array','minItems':1,'items':{'type':'string'}}}}}}}},'insufficient_evidence':{'type':'boolean'}}}
    prompt_path=write_json('llm_prompt.json',{'system':system_prompt,'user_template':'질문 + 익명 카드별 evidence_id/issuer/card_name/heading/text','forbidden':['system label','OLD/STRUCT','score','rank','gold','qualification','claim IDs','source path']}); schema_path=write_json('llm_response_schema.json',schema)
    requests=[]
    for rid in dataset_seal['seal_core']['record_ids']:
        for system in ('OLD','STRUCT'):
            groups=[]
            for p in sorted((x for x in packages if x['record_id']==rid and x['system']==system),key=lambda x:x['card_rank']): groups.append({'issuer':p['evidence'][0]['issuer'],'card_name':p['evidence'][0]['card_name'],'evidence':p['evidence']})
            user_payload={'query':query_by_id[rid],'card_evidence_groups':groups}; request={'model':'gpt-5.6-terra','reasoning':{'effort':'medium'},'tools':[],'store':False,'max_output_tokens':1200,'input':[{'role':'system','content':system_prompt},{'role':'user','content':canon(user_payload)}],'text':{'format':{'type':'json_schema','name':'card_answer','strict':True,'schema':schema}}}; requests.append({'payload_id':text_sha(rid+'|'+system)[:16],'record_id':rid,'internal_system':system,'request':request,'request_sha256':text_sha(canon(request))})
    if len(requests)!=60: raise RuntimeError('LLM payload count')
    payload_path=OUT/'llm_payloads.jsonl'; atomic_write_text(payload_path,''.join(canon(x)+'\n' for x in requests))
    enc=tiktoken.get_encoding('cl100k_base'); input_tokens=[len(enc.encode(canon(r['request']['input']))) for r in requests]
    llm_core={'schema_version':'llm_payload_approval_core_v1','candidate_core_sha256':guard['candidate_core_sha256'],'label_free_freeze_sha256':freeze_doc['freeze_sha256'],'retrieval_metrics_sha256':sha(metrics_path),'payloads_sha256':sha(payload_path),'prompt_sha256':sha(prompt_path),'schema_sha256':sha(schema_path),'model':'gpt-5.6-terra','reasoning_effort':'medium','request_cap':60,'input_token_estimate_cl100k':sum(input_tokens),'max_output_tokens_per_request':1200,'total_output_token_cap':72000,'tools':0,'store':False,'retries':0,'transmitted_allowlist':['query','issuer','card_name','heading','evidence_id','evidence_text'],'transmitted_forbidden':['system package label','rank','score','gold','qualification','atomic claim ID','source path','evaluation metric','secret'],'cost_usd_estimate':None,'cost_status':'unconfirmed_current_official_price_not_checked_offline','authorized':False}
    llm_approval_sha=text_sha(canon(llm_core)); write_json('llm_payload_approval.json',{'approval_core':llm_core,'approval_core_sha256':llm_approval_sha,'status':'awaiting_separate_user_approval','authorized':False})
    write_json('execution_final_status.json',{'status':'retrieval_scored_llm_payload_awaiting_approval','query_embedding_approval_sha256':guard['approval_core_sha256'],'execution_guard_sha256':guard_doc['execution_guard_sha256'],'label_free_freeze_sha256':freeze_doc['freeze_sha256'],'llm_payload_approval_sha256':llm_approval_sha,'api_requests_embedding':1,'llm_requests':0,'network_other':0})
    final_marker='\n## 검색 실행 완료 및 다음 승인 경계'; readme_final=(OUT/'README.md').read_text().split(final_marker,1)[0].replace('# 28 통합 holdout 후보 실행 전 봉인','# 28 통합 holdout 후보 검색 실행 및 LLM payload 승인 대기').replace('현재 상태는 `sealed_awaiting_query_embedding_approval`이며 `authorized=false`다.','검색/BGE/evidence projection은 완료됐고 LLM payload는 별도 승인 전 `authorized=false`다.').replace('- 실행: API/network/GPU/retrieval/BGE/LLM/new embedding 모두 0','- 실행 이력: 승인 query embedding 1회(현재 재개 run 0), local BGE 834 pairs, LLM 요청 0')
    readme_final += final_marker+f'''\n\n- OLD: Card Recall@3 {summary_by['OLD']['card_recall_at_3']:.6f}, Supported-card Recall@3 {summary_by['OLD']['supported_card_recall_at_3']:.6f}, Required Claim Coverage {summary_by['OLD']['required_claim_coverage']:.6f}, negative 4/4\n- STRUCT: Card Recall@3 {summary_by['STRUCT']['card_recall_at_3']:.6f}, Supported-card Recall@3 {summary_by['STRUCT']['supported_card_recall_at_3']:.6f}, Required Claim Coverage {summary_by['STRUCT']['required_claim_coverage']:.6f}, negative 4/4\n- paired STRUCT vs OLD: win {wlt.get('win',0)} / loss {wlt.get('loss',0)} / tie {wlt.get('tie',0)}; critical retrieval regression O07\n- 판정: retrieval-only 결과로 winner를 정하지 않으며, sealed scoring contract의 LLM answer gate 평가를 기다린다.\n- 다음 승인 범위: gpt-5.6-terra, reasoning medium, 60 requests, cl100k input estimate {sum(input_tokens)} tokens, output cap 72,000 tokens, retries 0, tools 0, store false. Offline에서 현재 공식 단가를 확인하지 않아 비용은 미확정이다.\n'''
    atomic_write_text(OUT/'README.md',readme_final)
    # Bind final outputs while preserving the source/cache input hashes.
    final_names=['README.md','query_embedding_attempt.json','query_embedding_usage.json','query_embedding_response_provenance.json','query_embedding_stage_ledger.json','label_free_retrieval_bge_stage_ledger.json','label_free_rankings.jsonl','bge_pair_scores.jsonl','candidate_rankings_freeze.jsonl','evidence_packages_freeze.jsonl','structural_bundles_freeze.jsonl','local_bge_resources.json','label_free_execution_freeze.json','retrieval_metrics.csv','retrieval_summary.json','retrieval_paired.csv','retrieval_wlt.json','retrieval_decision.json','llm_prompt.json','llm_response_schema.json','llm_payloads.jsonl','llm_payload_approval.json','execution_final_status.json']
    manifest_final=json.loads((OUT/'run_manifest.json').read_text()); manifest_final['post_approval_outputs']={n:sha(OUT/n) for n in final_names}; manifest_final['label_free_freeze_sha256']=freeze_doc['freeze_sha256']; manifest_final['llm_payload_approval_sha256']=llm_approval_sha; write_json('run_manifest.json',manifest_final)
    integrity_final=json.loads((OUT/'integrity.json').read_text()); integrity_final['post_approval_status']='PASS_RETRIEVAL_SCORED_LLM_AWAITING_APPROVAL'; integrity_final['label_free_freeze_sha256']=freeze_doc['freeze_sha256']; integrity_final['llm_payload_approval_sha256']=llm_approval_sha; integrity_final['gold_loaded_only_after_label_free_freeze']=True; integrity_final['llm_api_requests']=0; integrity_final['final_readme_sha256']=sha(OUT/'README.md'); write_json('integrity.json',integrity_final)
    complete_post({'retrieval_rows':len(rows),'llm_payloads':len(requests),'llm_api_requests':0})
    print(json.dumps({'status':'POST_FREEZE_SCORING_COMPLETE','retrieval_wlt':dict(wlt),'llm_payloads':60,'llm_input_tokens_cl100k':sum(input_tokens),'llm_approval_core_sha256':llm_approval_sha},indent=2))


SKIP post-freeze scoring: RUN_APPROVED_28_POST_FREEZE=0


## LLM external execution and answer scoring

This append-only stage binds the approved 60-request payload set to a fail-closed runner, transport validator, atomic resume ledger, and an offline factual scorer. The approved payload core is unchanged. The new execution guard hashes the runner, validator, scorer, payload, prompt, schema, and sealed gold inputs before the OpenAI client can be imported.

In [1]:
# CPU/offline transport validator shared by the guarded runner.
from pathlib import Path
import hashlib, json, os, re, tempfile, time, unicodedata
ROOT=Path.cwd(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
if ROOT.name!='PickCardU': raise RuntimeError('repository root mismatch')
OUT=ROOT/'notebooks/data/28_integrated_holdout_candidate_evaluation'
NB=ROOT/'notebooks/28_integrated_holdout_candidate_evaluation.ipynb'
D27=ROOT/'notebooks/data/27_integrated_holdout_dataset'
canon=lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
def require(ok,message):
    if not ok: raise RuntimeError(message)
def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x.strip()]
def atomic_json(path,value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(prefix=path.name+'.',suffix='.tmp',dir=path.parent)
    try:
        with os.fdopen(fd,'w',encoding='utf-8') as f:
            f.write(json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2)+'\n'); f.flush(); os.fsync(f.fileno())
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp): os.unlink(tmp)
payload_rows=read_jsonl(OUT/'llm_payloads.jsonl')
require(len(payload_rows)==60 and len({x['payload_id'] for x in payload_rows})==60,'payload cardinality')
payload_by_id={x['payload_id']:x for x in payload_rows}
payload_context={}
for row in payload_rows:
    request=row['request']; require(set(request)=={'input','max_output_tokens','model','reasoning','store','text','tools'},'request keys')
    user=json.loads(request['input'][1]['content']); require(set(user)=={'query','card_evidence_groups'},'transmitted user keys')
    owners={}; texts={}; evidence_to_group={}
    for group in user['card_evidence_groups']:
        require(set(group)=={'issuer','card_name','evidence'},'group keys')
        key=(group['issuer'],group['card_name']); require(key not in owners,'duplicate transmitted group')
        ids=set()
        for evidence in group['evidence']:
            require(set(evidence)=={'issuer','card_name','heading','evidence_id','text'},'evidence keys')
            require(evidence['issuer']==group['issuer'] and evidence['card_name']==group['card_name'],'evidence ownership')
            eid=evidence['evidence_id']; require(eid not in texts,'duplicate evidence id')
            ids.add(eid); texts[eid]=evidence['text']; evidence_to_group[eid]=key
        require(ids,'empty evidence group'); owners[key]=ids
    payload_context[row['payload_id']]={'owners':owners,'texts':texts,'evidence_to_group':evidence_to_group,'record_id':row['record_id'],'system':row['internal_system']}
def validate_llm_response(payload_id,response):
    if payload_id not in payload_context:
        return {'payload_id':payload_id,'status':'format_failure','format_errors':['unknown_payload_id'],'semantic_errors':[]}
    if not isinstance(response,dict):
        return {'payload_id':payload_id,'status':'format_failure','format_errors':['response_not_object'],'semantic_errors':[]}
    fmt=[]; sem=[]
    if set(response)!={'summary','cards','insufficient_evidence'}: fmt.append('top_level_keys_not_exact')
    if not isinstance(response.get('summary'),str): fmt.append('summary_not_string')
    if not isinstance(response.get('cards'),list): fmt.append('cards_not_array')
    if not isinstance(response.get('insufficient_evidence'),bool): fmt.append('insufficient_not_boolean')
    if fmt: return {'payload_id':payload_id,'status':'format_failure','format_errors':fmt,'semantic_errors':[]}
    cards=response['cards']; owners=payload_context[payload_id]['owners']
    if len(cards)>3: fmt.append('cards_over_k3')
    seen=set()
    for card in cards:
        if not isinstance(card,dict) or set(card)!={'issuer','card_name','claims'}:
            fmt.append('card_shape_invalid'); continue
        key=(card['issuer'],card['card_name'])
        if not all(isinstance(x,str) and x.strip() for x in key): fmt.append('card_identity_invalid')
        if key in seen: sem.append('duplicate_card_identity')
        seen.add(key)
        owner_ids=owners.get(key)
        if owner_ids is None: sem.append('card_identity_not_in_transmitted_group')
        claims=card['claims']
        if not isinstance(claims,list): fmt.append('claims_not_array'); continue
        for claim in claims:
            if not isinstance(claim,dict) or set(claim)!={'text','citations'}:
                fmt.append('claim_shape_invalid'); continue
            if not isinstance(claim['text'],str) or not claim['text'].strip(): fmt.append('claim_text_empty')
            cites=claim['citations']
            if not isinstance(cites,list) or not cites or any(not isinstance(x,str) or not x for x in cites):
                fmt.append('claim_citations_invalid'); continue
            if owner_ids is None or not set(cites).issubset(owner_ids): sem.append('claim_citation_ownership_mismatch')
    if response['insufficient_evidence'] and cards: sem.append('insufficient_with_cards')
    if not response['insufficient_evidence'] and not cards: sem.append('answer_without_cards')
    status='format_failure' if fmt else ('semantic_validation_failure' if sem else 'transport_semantic_validation_pass')
    return {'payload_id':payload_id,'status':status,'format_errors':sorted(set(fmt)),'semantic_errors':sorted(set(sem)),'factual_correctness_scored':False}
first=payload_rows[0]; pid=first['payload_id']; ctx=payload_context[pid]; key=next(iter(ctx['owners'])); eid=next(iter(ctx['owners'][key]))
good={'summary':'근거 요약','cards':[{'issuer':key[0],'card_name':key[1],'claims':[{'text':'근거에서 확인되는 내용','citations':[eid]}]}],'insufficient_evidence':False}
bad_owner=json.loads(json.dumps(good)); bad_owner['cards'][0]['claims'][0]['citations']=['UNKNOWN']
bad_card=json.loads(json.dumps(good)); bad_card['cards'][0]['issuer']='UNKNOWN'
bad_shape={'summary':'x','cards':'x','insufficient_evidence':False}
insufficient={'summary':'근거 부족','cards':[],'insufficient_evidence':True}
def resume_action(state,record_exists):
    if record_exists: return 'skip_terminal'
    if state=='in_flight': return 'fail_closed'
    if state is None: return 'call_once'
    return 'fail_closed'
def audit_resume_ledger(ledger,rows,record_meta,guard_sha,require_complete=False):
    ids=[x['payload_id'] for x in rows]; row_by={x['payload_id']:x for x in rows}
    require(isinstance(ledger,dict) and ledger.get('schema_version')=='28_llm_resume_ledger_v1','ledger schema')
    require(ledger.get('execution_guard_sha256')==guard_sha and ledger.get('ordered_payload_ids')==ids,'ledger guard/order')
    states=ledger.get('states'); attempts=ledger.get('request_attempts_total')
    require(isinstance(states,dict) and set(states)<=set(ids),'ledger states coverage')
    require(isinstance(attempts,int) and not isinstance(attempts,bool) and 0<=attempts<=len(ids),'ledger attempt range')
    require(attempts==len(states),'ledger attempt/state count mismatch')
    require(set(record_meta)=={pid for pid,state in states.items() if state.get('state')=='terminal'},'record/terminal coverage mismatch')
    for pid,state in states.items():
        require(isinstance(state,dict) and state.get('state') in {'in_flight','terminal'},'ledger state invalid')
        if state['state']=='in_flight': require(pid not in record_meta,'in-flight record ambiguity')
        else:
            meta=record_meta[pid]; record=meta['record']; row=row_by[pid]
            require(meta['sha256']==state.get('record_sha256'),'record hash mismatch')
            require(record.get('payload_id')==pid and record.get('request_sha256')==row['request_sha256'],'record request association mismatch')
            require(record.get('outcome')==state.get('outcome') and record.get('outcome') in {'response_saved','request_error_terminal_no_retry'},'record outcome mismatch')
            validation=record.get('transport_validation')
            require(isinstance(validation,dict) and validation.get('payload_id')==pid and validation.get('status') in {'transport_semantic_validation_pass','semantic_validation_failure','format_failure'},'record validation association mismatch')
    if require_complete:
        require(attempts==len(ids) and set(states)==set(ids) and all(x.get('state')=='terminal' for x in states.values()),'ledger not terminal complete')
    return True
def audit_fails(fn):
    try: fn()
    except RuntimeError: return True
    return False
def select_llm_decision(old_eligible,struct_eligible,gap,wins,losses):
    if old_eligible and not struct_eligible: return 'OLD'
    if struct_eligible and not old_eligible: return 'STRUCT'
    if not old_eligible and not struct_eligible: return 'holdout_failed'
    if gap>=3 and wins>losses: return 'STRUCT'
    if gap<=-3 and losses>wins: return 'OLD'
    return 'inconclusive'
synthetic=[
    ('valid',validate_llm_response(pid,good),'transport_semantic_validation_pass'),
    ('ownership',validate_llm_response(pid,bad_owner),'semantic_validation_failure'),
    ('identity',validate_llm_response(pid,bad_card),'semantic_validation_failure'),
    ('shape',validate_llm_response(pid,bad_shape),'format_failure'),
    ('insufficient',validate_llm_response(pid,insufficient),'transport_semantic_validation_pass')]
require(all(result['status']==expected for _,result,expected in synthetic),'validator synthetic audit')
resume_synthetic=[('new',resume_action(None,False),'call_once'),('complete',resume_action('terminal',True),'skip_terminal'),('interrupted',resume_action('in_flight',False),'fail_closed'),('terminal_missing',resume_action('terminal',False),'fail_closed')]
require(all(actual==expected for _,actual,expected in resume_synthetic),'resume synthetic audit')
syn_row={'payload_id':'SYN','request_sha256':'req'}; syn_record={'payload_id':'SYN','request_sha256':'req','outcome':'response_saved','transport_validation':{'payload_id':'SYN','status':'transport_semantic_validation_pass'}}; syn_meta={'SYN':{'sha256':'recordsha','record':syn_record}}
syn_ledger={'schema_version':'28_llm_resume_ledger_v1','execution_guard_sha256':'guard','ordered_payload_ids':['SYN'],'states':{'SYN':{'state':'terminal','outcome':'response_saved','record_sha256':'recordsha'}},'request_attempts_total':1,'status':'complete'}
ledger_integrity_synthetic=[('valid_complete',audit_resume_ledger(syn_ledger,[syn_row],syn_meta,'guard',True),True),('corrupt_attempt_count',audit_fails(lambda:audit_resume_ledger({**syn_ledger,'request_attempts_total':0},[syn_row],syn_meta,'guard',True)),True),('cross_request_record',audit_fails(lambda:audit_resume_ledger(syn_ledger,[syn_row],{'SYN':{'sha256':'recordsha','record':{**syn_record,'request_sha256':'other'}}},'guard',True)),True)]
require(all(actual==expected for _,actual,expected in ledger_integrity_synthetic),'ledger integrity synthetic audit')
scorer_synthetic=[('old_only',select_llm_decision(True,False,0,0,0),'OLD'),('struct_only',select_llm_decision(False,True,0,0,0),'STRUCT'),('both_fail',select_llm_decision(False,False,0,0,0),'holdout_failed'),('struct_gap',select_llm_decision(True,True,3,4,1),'STRUCT'),('tie',select_llm_decision(True,True,1,2,2),'inconclusive')]
require(all(actual==expected for _,actual,expected in scorer_synthetic),'scorer decision synthetic audit')
validator_contract={'schema_version':'28_llm_transport_validator_v1','response_count':60,'status_meaning':{'transport_semantic_validation_pass':'shape, transmitted card identity, citation ownership, and answer-state checks only; not factual correctness','semantic_validation_failure':'stored and scored; does not abort batch','format_failure':'stored and scored; does not abort batch'},'request_failure_behavior':'terminal record; never retried by this approved single run','required_offline_followup':'llm_answer_scores.jsonl','api_network_current':0}
atomic_json(OUT/'llm_response_validation_contract.json',validator_contract)
atomic_json(OUT/'llm_validator_synthetic_audit.json',{'status':'PASS','cases':[{'case':n,'result':r,'expected':e} for n,r,e in synthetic],'external_api_network_gpu_model':0})
atomic_json(OUT/'llm_resume_synthetic_audit.json',{'status':'PASS','cases':[{'case':n,'actual':a,'expected':e} for n,a,e in resume_synthetic],'ledger_integrity_cases':[{'case':n,'actual':a,'expected':e} for n,a,e in ledger_integrity_synthetic],'request_retries':0,'external_api_network_gpu_model':0})
atomic_json(OUT/'llm_scorer_synthetic_audit.json',{'status':'PASS','cases':[{'case':n,'actual':a,'expected':e} for n,a,e in scorer_synthetic],'candidate_result_constants_used':False,'external_api_network_gpu_model':0})
print({'llm_validator_preflight':'PASS','payloads':60,'validator_cases':len(synthetic),'resume_cases':len(resume_synthetic)+len(ledger_integrity_synthetic),'scorer_cases':len(scorer_synthetic)})

{'llm_validator_preflight': 'PASS', 'payloads': 60, 'validator_cases': 5, 'resume_cases': 7, 'scorer_cases': 5}


In [2]:
# Bind the already-approved payload core to exact runner/validator/scorer source hashes.
require(os.environ.get('RUN_APPROVED_28_LLM','0') in {'0','1'},'RUN_APPROVED_28_LLM must be 0 or 1')
APPROVED_LLM_CORE='ef74af3eefd2195d6e4b9d5bd33b0f60f0f6629253a3c336bd9929d15ddd0c57'
doc=json.loads(NB.read_text(encoding='utf-8'))
def cell_sha(cell_id):
    cell=next((c for c in doc['cells'] if c.get('id')==cell_id),None); require(cell is not None,'missing cell '+cell_id)
    source=''.join(cell.get('source',[])) if isinstance(cell.get('source',[]),list) else cell.get('source','')
    return hashlib.sha256(canon({'cell_id':cell_id,'source_text':source}).encode()).hexdigest()
validator_sha=cell_sha('28-llm-validator-source'); runner_sha=cell_sha('28-llm-external-runner'); scorer_sha=cell_sha('28-llm-factual-scoring')
approval=json.loads((OUT/'llm_payload_approval.json').read_text())
require(hashlib.sha256(canon(approval['approval_core']).encode()).hexdigest()==approval['approval_core_sha256']==APPROVED_LLM_CORE,'approved LLM core mismatch')
core=approval['approval_core']
require(core['request_cap']==60 and core['input_token_estimate_cl100k']==466486,'approved input cap mismatch')
require(core['total_output_token_cap']==72000 and core['max_output_tokens_per_request']==1200,'approved output cap mismatch')
require(core['model']=='gpt-5.6-terra' and core['reasoning_effort']=='medium','approved model mismatch')
require(core['tools']==0 and core['store'] is False and core['retries']==0,'approved API settings mismatch')
require(sha(OUT/'llm_payloads.jsonl')==core['payloads_sha256'],'payload hash mismatch')
require(sha(OUT/'llm_prompt.json')==core['prompt_sha256'],'prompt hash mismatch')
require(sha(OUT/'llm_response_schema.json')==core['schema_sha256'],'schema hash mismatch')
require(sha(OUT/'retrieval_metrics.csv')==core['retrieval_metrics_sha256'],'retrieval metric hash mismatch')
sealed=json.loads((D27/'dataset_seal.json').read_text()); manifest27=json.loads((D27/'draft_dataset_manifest.json').read_text())
require(sealed['seal_core']['status']=='sealed_not_evaluated' and sealed['seal_core']['record_count']==30,'dataset seal status/count')
gold_files={'dataset_seal.json':sha(D27/'dataset_seal.json'),'master_gold.jsonl':sha(D27/'master_gold.jsonl'),'qualification_by_card.jsonl':sha(D27/'qualification_by_card.jsonl'),'atomic_claims.json':sha(D27/'atomic_claims.json'),'candidate_projection.jsonl':sha(D27/'candidate_projection.jsonl'),'source_components.jsonl':sha(D27/'source_components.jsonl'),'draft_scoring_contract.json':sha(D27/'draft_scoring_contract.json')}
for name,digest in gold_files.items():
    if name in manifest27['artifacts']: require(digest==manifest27['artifacts'][name],'sealed gold hash mismatch '+name)
require(gold_files['master_gold.jsonl']==sealed['seal_core']['hashes']['gold'],'gold seal mismatch')
require(gold_files['atomic_claims.json']==sealed['seal_core']['hashes']['claims'],'claim seal mismatch')
require(gold_files['candidate_projection.jsonl']==sealed['seal_core']['hashes']['projection'],'projection seal mismatch')
require(gold_files['draft_scoring_contract.json']==sealed['seal_core']['hashes']['scoring'],'scoring seal mismatch')
ordered_request_sha=[x['request_sha256'] for x in payload_rows]
require(len(ordered_request_sha)==60 and all(hashlib.sha256(canon(x['request']).encode()).hexdigest()==x['request_sha256'] for x in payload_rows),'request sha mismatch')
response_freeze=None; ledger_path=OUT/'llm_resume_ledger.json'; records_dir=OUT/'llm_response_records'; response_guard_freeze_path=OUT/'llm_response_execution_guard_freeze.json'
if ledger_path.exists() or records_dir.exists():
    require(ledger_path.exists() and records_dir.is_dir(),'partial response execution state')
    ledger=json.loads(ledger_path.read_text()); record_meta={}
    for path in sorted(records_dir.glob('*.json')):
        record=json.loads(path.read_text()); require(path.stem==record.get('payload_id') and path.stem not in record_meta,'response freeze record ID')
        record_meta[path.stem]={'sha256':sha(path),'record':record}
    if response_guard_freeze_path.exists(): response_guard_doc=json.loads(response_guard_freeze_path.read_text())
    else: response_guard_doc=json.loads((OUT/'llm_execution_guard.json').read_text())
    response_guard_sha=response_guard_doc['execution_guard_sha256']; response_guard_core=response_guard_doc['guard_core']
    require(hashlib.sha256(canon(response_guard_core).encode()).hexdigest()==response_guard_sha==ledger.get('execution_guard_sha256'),'completed response guard mismatch')
    require(response_guard_core['approved_llm_core_sha256']==APPROVED_LLM_CORE and response_guard_core['validator_source_sha256']==validator_sha and response_guard_core['runner_source_sha256']==runner_sha,'completed response source/approval mismatch')
    require(audit_resume_ledger(ledger,payload_rows,record_meta,response_guard_sha,True),'completed response ledger audit')
    if not response_guard_freeze_path.exists(): atomic_json(response_guard_freeze_path,response_guard_doc)
    response_freeze={'response_execution_guard_sha256':response_guard_sha,'resume_ledger_sha256':sha(ledger_path),'record_hashes':{pid:record_meta[pid]['sha256'] for pid in sorted(record_meta)},'terminal_records':60,'request_attempts_total':60}
guard_core={'schema_version':'28_llm_execution_guard_v1','approved_llm_core_sha256':APPROVED_LLM_CORE,'candidate_core_sha256':core['candidate_core_sha256'],'label_free_freeze_sha256':core['label_free_freeze_sha256'],'payloads_sha256':core['payloads_sha256'],'prompt_sha256':core['prompt_sha256'],'schema_sha256':core['schema_sha256'],'retrieval_metrics_sha256':core['retrieval_metrics_sha256'],'gold_input_hashes':gold_files,'ordered_request_sha256':ordered_request_sha,'validator_source_sha256':validator_sha,'runner_source_sha256':runner_sha,'scorer_source_sha256':scorer_sha,'completed_response_freeze':response_freeze,'response_count':60,'input_token_cap':466486,'output_token_cap':72000,'max_output_tokens_per_request':1200,'sdk_retries':0,'required_env':'APPROVED_28_LLM_EXECUTION_GUARD_SHA256','resume_policy':'terminal atomic record per payload; completed or request-error payloads never called again; unresolved in-flight attempt fails closed'}
guard_sha=hashlib.sha256(canon(guard_core).encode()).hexdigest()
atomic_json(OUT/'llm_execution_guard.json',{'guard_core':guard_core,'execution_guard_sha256':guard_sha})
atomic_json(OUT/'llm_external_preflight.json',{'status':'PASS_AWAITING_MAIN_EXECUTION','approved_llm_core_sha256':APPROVED_LLM_CORE,'execution_guard_sha256':guard_sha,'validator_source_sha256':validator_sha,'runner_source_sha256':runner_sha,'scorer_source_sha256':scorer_sha,'requests_current':0,'network_api_gpu_model_current':0,'approval_scope_changed':False,'new_user_reapproval_required':False})
print({'llm_external_preflight':'PASS','approved_core':APPROVED_LLM_CORE,'execution_guard':guard_sha,'requests_current':0})

{'llm_external_preflight': 'PASS', 'approved_core': 'ef74af3eefd2195d6e4b9d5bd33b0f60f0f6629253a3c336bd9929d15ddd0c57', 'execution_guard': 'dd40c96b0bcc0b7c4be561238436a2fe5037338909568e1992cd74c5bf0a09d8', 'requests_current': 0}


In [ ]:
# Guarded Responses API runner: atomic per-request records and fail-closed resume.
def ext_require(ok,message):
    if not ok: raise RuntimeError(message)
ext_require(os.environ.get('RUN_APPROVED_28_LLM')=='1','LLM execution is not enabled')
guard_doc=json.loads((OUT/'llm_execution_guard.json').read_text()); g=guard_doc['guard_core']
ext_require(hashlib.sha256(canon(g).encode()).hexdigest()==guard_doc['execution_guard_sha256']==os.environ.get('APPROVED_28_LLM_EXECUTION_GUARD_SHA256',''),'execution guard mismatch')
ext_require(g['approved_llm_core_sha256']=='ef74af3eefd2195d6e4b9d5bd33b0f60f0f6629253a3c336bd9929d15ddd0c57','approval core mismatch')
current_doc=json.loads(NB.read_text(encoding='utf-8'))
def ext_cell_sha(cell_id):
    cell=next((c for c in current_doc['cells'] if c.get('id')==cell_id),None); ext_require(cell is not None,'missing guarded cell')
    source=''.join(cell.get('source',[])) if isinstance(cell.get('source',[]),list) else cell.get('source','')
    return hashlib.sha256(canon({'cell_id':cell_id,'source_text':source}).encode()).hexdigest()
ext_require(ext_cell_sha('28-llm-validator-source')==g['validator_source_sha256'],'validator source changed')
ext_require(ext_cell_sha('28-llm-external-runner')==g['runner_source_sha256'],'runner source changed')
ext_require(ext_cell_sha('28-llm-factual-scoring')==g['scorer_source_sha256'],'scorer source changed')
ext_require(sha(OUT/'llm_payloads.jsonl')==g['payloads_sha256'] and sha(OUT/'llm_prompt.json')==g['prompt_sha256'] and sha(OUT/'llm_response_schema.json')==g['schema_sha256'],'payload contract changed')
for name,digest in g['gold_input_hashes'].items(): ext_require(sha(D27/name)==digest,'gold input changed: '+name)
ext_require(len(payload_rows)==g['response_count']==60,'response count mismatch')
records_dir=OUT/'llm_response_records'; records_dir.mkdir(exist_ok=True); ledger_path=OUT/'llm_resume_ledger.json'
if ledger_path.exists(): ledger=json.loads(ledger_path.read_text())
else: ledger={'schema_version':'28_llm_resume_ledger_v1','execution_guard_sha256':guard_doc['execution_guard_sha256'],'ordered_payload_ids':[x['payload_id'] for x in payload_rows],'states':{},'request_attempts_total':0,'status':'ready'}
def load_record_meta():
    meta={}
    for path in sorted(records_dir.glob('*.json')):
        record=json.loads(path.read_text()); ext_require(path.stem==record.get('payload_id') and path.stem not in meta,'record filename or duplicate ID mismatch')
        meta[path.stem]={'sha256':sha(path),'record':record}
    return meta
audit_resume_ledger(ledger,payload_rows,load_record_meta(),guard_doc['execution_guard_sha256'])
ext_require(not any(x.get('state')=='in_flight' for x in ledger['states'].values()),'unresolved in-flight request; fail closed without retry')
def save_ledger(): atomic_json(ledger_path,ledger)
from openai import OpenAI
client=OpenAI(max_retries=0)
for item in payload_rows:
    audit_resume_ledger(ledger,payload_rows,load_record_meta(),guard_doc['execution_guard_sha256'])
    pid=item['payload_id']; record_path=records_dir/(pid+'.json')
    if record_path.exists():
        continue
    ext_require(pid not in ledger['states'],'nonterminal ledger state prevents retry')
    ext_require(ledger['request_attempts_total']<g['response_count'],'request cap exhausted before API call')
    ledger['states'][pid]={'state':'in_flight','started_unix':time.time(),'request_sha256':item['request_sha256']}; ledger['request_attempts_total']+=1; ledger['status']='running'; save_ledger()
    audit_resume_ledger(ledger,payload_rows,load_record_meta(),guard_doc['execution_guard_sha256'])
    started=time.perf_counter()
    try:
        response=client.responses.create(**item['request']); raw=response.model_dump(mode='json'); latency=time.perf_counter()-started
        texts=[p.get('text','') for out in raw.get('output',[]) if out.get('type')=='message' for p in out.get('content',[]) if p.get('type')=='output_text']
        try: parsed=json.loads('\n'.join(texts)) if texts else None
        except Exception: parsed=None
        validation=validate_llm_response(pid,parsed)
        if raw.get('status')!='completed' or parsed is None: validation={'payload_id':pid,'status':'format_failure','format_errors':['provider_incomplete_refusal_or_unparseable_json'],'semantic_errors':[],'factual_correctness_scored':False}
        record={'payload_id':pid,'request_sha256':item['request_sha256'],'outcome':'response_saved','raw_response':raw,'parsed_response':parsed,'transport_validation':validation,'usage':raw.get('usage') or {},'latency_seconds':latency}
    except Exception as exc:
        latency=time.perf_counter()-started
        record={'payload_id':pid,'request_sha256':item['request_sha256'],'outcome':'request_error_terminal_no_retry','request_error_type':type(exc).__name__,'raw_response':None,'parsed_response':None,'transport_validation':{'payload_id':pid,'status':'format_failure','format_errors':['request_error'],'semantic_errors':[],'factual_correctness_scored':False},'usage':{},'latency_seconds':latency}
    atomic_json(record_path,record); ledger['states'][pid]={'state':'terminal','outcome':record['outcome'],'record_sha256':sha(record_path)}; save_ledger(); audit_resume_ledger(ledger,payload_rows,load_record_meta(),guard_doc['execution_guard_sha256'])
ledger['status']='complete' if len(ledger['states'])==60 and all(x['state']=='terminal' for x in ledger['states'].values()) else 'incomplete'; save_ledger()
audit_resume_ledger(ledger,payload_rows,load_record_meta(),guard_doc['execution_guard_sha256'],True)
print({'llm_runner_status':ledger['status'],'terminal_records':len(ledger['states']),'request_attempts_total':ledger['request_attempts_total']})

In [3]:
# Self-contained CPU/offline answer scoring from atomic response records.
if os.environ.get('RUN_APPROVED_28_LLM_SCORING','0')!='1':
    print({'llm_factual_scoring':'SKIPPED','reason':'RUN_APPROVED_28_LLM_SCORING is not 1'})
else:
    guard_doc=json.loads((OUT/'llm_execution_guard.json').read_text()); g=guard_doc['guard_core']
    approval=json.loads((OUT/'llm_payload_approval.json').read_text())
    require(hashlib.sha256(canon(approval['approval_core']).encode()).hexdigest()==approval['approval_core_sha256']==g['approved_llm_core_sha256'],'scoring approval mismatch')
    require(hashlib.sha256(canon(g).encode()).hexdigest()==guard_doc['execution_guard_sha256'],'scoring guard mismatch')
    score_doc=json.loads(NB.read_text(encoding='utf-8'))
    def score_cell_sha(cell_id):
        cell=next((c for c in score_doc['cells'] if c.get('id')==cell_id),None); require(cell is not None,'missing scoring-bound cell')
        source=''.join(cell.get('source',[])) if isinstance(cell.get('source',[]),list) else cell.get('source','')
        return hashlib.sha256(canon({'cell_id':cell_id,'source_text':source}).encode()).hexdigest()
    require(score_cell_sha('28-llm-validator-source')==g['validator_source_sha256'] and score_cell_sha('28-llm-external-runner')==g['runner_source_sha256'] and score_cell_sha('28-llm-factual-scoring')==g['scorer_source_sha256'],'scoring-bound source changed')
    require(sha(OUT/'llm_payloads.jsonl')==g['payloads_sha256'] and sha(OUT/'llm_prompt.json')==g['prompt_sha256'] and sha(OUT/'llm_response_schema.json')==g['schema_sha256'],'scoring payload contract changed')
    for name,digest in g['gold_input_hashes'].items(): require(sha(D27/name)==digest,'scoring gold changed')
    response_freeze=g.get('completed_response_freeze'); require(isinstance(response_freeze,dict) and response_freeze.get('terminal_records')==60 and response_freeze.get('request_attempts_total')==60,'missing completed response freeze')
    ledger=json.loads((OUT/'llm_resume_ledger.json').read_text()); record_paths=sorted((OUT/'llm_response_records').glob('*.json')); require(len(record_paths)==60,'need 60 terminal records')
    record_meta={}
    for path in record_paths:
        record=json.loads(path.read_text()); require(path.stem==record.get('payload_id') and path.stem not in record_meta,'scoring record filename/ID')
        record_meta[path.stem]={'sha256':sha(path),'record':record}
    require(sha(OUT/'llm_resume_ledger.json')==response_freeze['resume_ledger_sha256'] and {pid:record_meta[pid]['sha256'] for pid in sorted(record_meta)}==response_freeze['record_hashes'],'scoring response freeze hash mismatch')
    require(audit_resume_ledger(ledger,payload_rows,record_meta,response_freeze['response_execution_guard_sha256'],True),'scoring ledger audit')
    records={pid:meta['record'] for pid,meta in record_meta.items()}; require(set(records)==set(payload_by_id),'response ID coverage')
    packages=read_jsonl(OUT/'evidence_packages_freeze.jsonl'); gold_rows=read_jsonl(D27/'master_gold.jsonl'); gold_by={x['record_id']:x for x in gold_rows}
    claims_doc=json.loads((D27/'atomic_claims.json').read_text()); claims={x['atomic_claim_id']:x for x in claims_doc['claims'] if x['evaluation_status']=='active_evaluation'}
    projections=read_jsonl(D27/'candidate_projection.jsonl'); atomic_projection_rows=[x for x in projections if x.get('projection_scope')=='atomic_claim']; negative_projection_rows=[x for x in projections if x.get('projection_scope')=='negative_record']
    require(len(projections)==104 and len(atomic_projection_rows)==96 and len(negative_projection_rows)==8,'projection scope cardinality')
    require(all(set(('corpus','card_id','atomic_claim_id'))<=set(x) for x in atomic_projection_rows),'atomic projection canonical top-level join key')
    require(all('record_id' in x and 'card_id' not in x and 'atomic_claim_id' not in x for x in negative_projection_rows),'negative projection schema')
    proj={(x['corpus'],x['card_id'],x['atomic_claim_id']):x for x in atomic_projection_rows}; require(len(proj)==96,'atomic projection join uniqueness')
    source_components={x['source_component_id']:x for x in read_jsonl(D27/'source_components.jsonl')}; require(len(source_components)==78,'source component cardinality')
    component_source_text={}
    for cid,row in source_components.items():
        source_path=ROOT/row['source_path']; raw=source_path.read_bytes(); require(hashlib.sha256(raw).hexdigest()==row['source_sha256'],'component source hash')
        excerpt_bytes=raw[row['byte_start']:row['byte_end']]; require(hashlib.sha256(excerpt_bytes).hexdigest()==row['excerpt_sha256'],'component excerpt hash')
        excerpt=excerpt_bytes.decode('utf-8'); require(raw.decode('utf-8')[row['cp_start']:row['cp_end']]==excerpt,'component byte/codepoint interval')
        component_source_text[cid]=excerpt
    scoring_contract=json.loads((D27/'draft_scoring_contract.json').read_text())
    canonical_card={'BC':'BC','NH':'NH','hana':'HANA','hyundai':'HYUNDAI','ibk':'IBK','kookmin':'KB','lotte':'LOTTE','samsung':'SAMSUNG','shinhan':'SHINHAN','woori':'WOORI'}
    norm=lambda x:' '.join(unicodedata.normalize('NFKC',str(x)).casefold().split())
    tok=lambda x:set(re.findall(r'[0-9a-z가-힣%]+',norm(x))); nums=lambda x:set(re.findall(r'\d+(?:[.,]\d+)?\s*(?:%|만원|천원|원|개월|포인트|마일|회|일|년|점)?',norm(x)))
    def graph_units(graph):
        units=[]
        for key,value in graph.items():
            if key=='all_of': units.extend([[leaf[0]] for leaf in value])
            elif key=='any_of': units.append([leaf[0] for leaf in value])
            else: raise RuntimeError('unsupported graph operator')
        return units
    def evidence_claim_support(system,card_id,claim_id,cited_chunk_ids,claim_text):
        claim=claims.get(claim_id); projection=proj.get((system,card_id,claim_id))
        if claim is None or projection is None: return False
        component_map=projection['payload_evidence_projection']['eligible_component_chunk_ids']; required=claim['required_component_ids']
        if not required or any(not (set(component_map.get(cid,[]))&cited_chunk_ids) for cid in required): return False
        truth=' '.join(component_source_text[cid] for cid in required); a,b=tok(claim_text),tok(truth)
        return bool(a) and len(a&b)/len(a)>=0.25 and nums(claim_text)<=nums(truth)
    score_rows=[]
    for item in payload_rows:
        pid=item['payload_id']; rid=item['record_id']; system=item['internal_system']; rec=records[pid]; response=rec['parsed_response']; transport=rec['transport_validation']; gold=gold_by[rid]; expected=set(gold['expected_card_ids'])
        ctx=payload_context[pid]; wire_to_card={}; evidence_to_chunk={}; evidence_to_card={}
        for (issuer,card_name),ids in ctx['owners'].items():
            matches=[p for p in packages if p['record_id']==rid and p['system']==system and p['evidence'][0]['issuer']==issuer and p['evidence'][0]['card_name']==card_name]
            require(len(matches)==1,'wire package mapping'); package=matches[0]; card_id=canonical_card[package['card_key'].split('/',1)[0]]; wire_to_card[(issuer,card_name)]=card_id
            for evidence,chunk_id in zip(package['evidence'],package['selected_chunk_ids']): evidence_to_chunk[evidence['evidence_id']]=chunk_id; evidence_to_card[evidence['evidence_id']]=card_id
        cards=response.get('cards',[]) if isinstance(response,dict) and isinstance(response.get('cards'),list) else []
        returned=[]; supported_by_card={}; total_citations=0; owned_citations=0; unsupported_claims=0; format_errors=int(transport['status']=='format_failure'); semantic_errors=int(transport['status']=='semantic_validation_failure')
        for card in cards[:3]:
            if not isinstance(card,dict): format_errors+=1; continue
            key=(card.get('issuer'),card.get('card_name')); card_id=wire_to_card.get(key)
            if card_id is not None: returned.append(card_id)
            for claim_row in card.get('claims',[]) if isinstance(card.get('claims'),list) else []:
                cites=claim_row.get('citations',[]) if isinstance(claim_row,dict) else []; total_citations+=len(cites)
                owned={eid for eid in cites if evidence_to_card.get(eid)==card_id}; owned_citations+=len(owned); cited_chunks={evidence_to_chunk[eid] for eid in owned if eid in evidence_to_chunk}; matched=set()
                if card_id is not None:
                    for claim_id,claim in claims.items():
                        if claim['card_id']==card_id and evidence_claim_support(system,card_id,claim_id,cited_chunks,claim_row.get('text','')): matched.add(claim_id)
                if not matched: unsupported_claims+=1
                supported_by_card.setdefault(card_id,set()).update(matched)
        duplicate_cards=len(returned)-len(set(returned)); returned=list(dict.fromkeys(returned)); supported_cards=set(); supported_units=0; total_units=0
        for card_id,graph in gold['qualification_graphs'].items():
            units=graph_units(graph); total_units+=len(units); count=sum(any(cid in supported_by_card.get(card_id,set()) for cid in unit) for unit in units); supported_units+=count
            if count==len(units): supported_cards.add(card_id)
        positive=bool(expected); insufficient=bool(response.get('insufficient_evidence')) if isinstance(response,dict) else True
        if positive:
            card_precision=len(expected&set(returned))/len(returned) if returned else 0.0; card_recall=len(expected&set(returned))/len(expected); supported_recall=len(expected&set(returned)&supported_cards)/len(expected); claim_coverage=supported_units/total_units if total_units else 0.0; negative_correctness=None
        else:
            card_precision=card_recall=supported_recall=claim_coverage=None; negative_correctness=int(not returned and insufficient)
        citation_ownership=owned_citations/total_citations if total_citations else (0.0 if positive else None); wrong_cards=len(set(returned)-expected); negative_false=int(not positive and bool(returned)); critical_errors=wrong_cards+negative_false+unsupported_claims
        e2e=int((positive and set(returned)==expected and supported_recall==1 and claim_coverage==1 and citation_ownership==1 and not insufficient and format_errors+semantic_errors+critical_errors+duplicate_cards==0) or (not positive and negative_correctness==1 and format_errors==0))
        score_rows.append({'payload_id':pid,'record_id':rid,'system':system,'positive':positive,'transport_status':transport['status'],'returned_card_ids':returned,'card_precision_at_3':card_precision,'card_recall_at_3':card_recall,'supported_card_recall_at_3':supported_recall,'required_claim_coverage':claim_coverage,'citation_ownership':citation_ownership,'negative_correctness':negative_correctness,'end_to_end_exact_success':e2e,'format_error_count':format_errors,'semantic_validation_error_count':semantic_errors,'critical_error_count':critical_errors,'wrong_card_error_count':wrong_cards,'negative_false_recommendation_count':negative_false,'unsupported_claim_count':unsupported_claims,'duplicate_card_count':duplicate_cards})
    require(len(score_rows)==60,'score rows'); score_path=OUT/'llm_answer_scores.jsonl'; score_path.write_text(''.join(canon(x)+'\n' for x in score_rows),encoding='utf-8')
    summaries=[]
    for system in ('OLD','STRUCT'):
        rows=[x for x in score_rows if x['system']==system]; pos=[x for x in rows if x['positive']]; neg=[x for x in rows if not x['positive']]
        summaries.append({'system':system,'positive_records':26,'negative_records':4,'card_precision_at_3':sum(x['card_precision_at_3'] for x in pos)/26,'card_recall_at_3':sum(x['card_recall_at_3'] for x in pos)/26,'supported_card_recall_at_3':sum(x['supported_card_recall_at_3'] for x in pos)/26,'required_claim_coverage':sum(x['required_claim_coverage'] for x in pos)/26,'citation_ownership':sum(x['citation_ownership'] for x in pos)/26,'negative_correctness_count':sum(x['negative_correctness'] for x in neg),'end_to_end_exact_success_count':sum(x['end_to_end_exact_success'] for x in rows),'derived_error_total':sum(x['format_error_count']+x['semantic_validation_error_count']+x['critical_error_count']+x['duplicate_card_count'] for x in rows)})
    by={x['system']:x for x in summaries}; absolute=scoring_contract['selection_contract']['absolute']
    eligible={s:(by[s]['end_to_end_exact_success_count']>=absolute['end_to_end_exact_success_count_min'] and by[s]['supported_card_recall_at_3']>=absolute['supported_card_recall_at_3_macro_min'] and by[s]['required_claim_coverage']>=absolute['required_claim_coverage_macro_min'] and by[s]['negative_correctness_count']==4 and by[s]['citation_ownership']==1 and by[s]['derived_error_total']==0) for s in ('OLD','STRUCT')}
    paired=[]; wins=losses=ties=0
    for rid in [x['record_id'] for x in gold_rows]:
        old=next(x for x in score_rows if x['record_id']==rid and x['system']=='OLD'); new=next(x for x in score_rows if x['record_id']==rid and x['system']=='STRUCT'); delta=new['end_to_end_exact_success']-old['end_to_end_exact_success']; outcome='win' if delta>0 else ('loss' if delta<0 else 'tie'); wins+=outcome=='win'; losses+=outcome=='loss'; ties+=outcome=='tie'; paired.append({'record_id':rid,'old':old['end_to_end_exact_success'],'struct':new['end_to_end_exact_success'],'delta':delta,'outcome':outcome})
    gap=by['STRUCT']['end_to_end_exact_success_count']-by['OLD']['end_to_end_exact_success_count']; decision=select_llm_decision(eligible['OLD'],eligible['STRUCT'],gap,wins,losses)
    atomic_json(OUT/'llm_answer_summary.json',{'systems':summaries,'eligible':eligible,'paired_wlt':{'wins':wins,'losses':losses,'ties':ties},'decision':decision,'single_run_no_population_generalization':True,'scorer_limitation':'Deterministic citation-to-sealed-component plus lexical/numeric claim matching; no external LLM judge. Paraphrases can be false negatives.'})
    (OUT/'llm_answer_paired.jsonl').write_text(''.join(canon(x)+'\n' for x in paired),encoding='utf-8')
    usage={'input_tokens':sum(int(x.get('usage',{}).get('input_tokens',0)) for x in records.values()),'output_tokens':sum(int(x.get('usage',{}).get('output_tokens',0)) for x in records.values()),'reasoning_tokens':sum(int(x.get('usage',{}).get('output_tokens_details',{}).get('reasoning_tokens',0)) for x in records.values()),'cached_input_tokens':sum(int(x.get('usage',{}).get('input_tokens_details',{}).get('cached_tokens',0)) for x in records.values()),'total_tokens':sum(int(x.get('usage',{}).get('total_tokens',0)) for x in records.values())}
    latency_values=[float(x['latency_seconds']) for x in records.values()]; usage_cost={'schema_version':'28_llm_actual_usage_v1','responses':60,'usage':usage,'latency_seconds_total':sum(latency_values),'latency_seconds_min':min(latency_values),'latency_seconds_max':max(latency_values),'actual_provider_cost_usd':None,'cost_status':'unconfirmed_current_official_price_not_checked_in_offline_scoring','external_requests_historical':60,'external_requests_current_scoring_run':0}
    atomic_json(OUT/'llm_usage_cost.json',usage_cost)
    atomic_json(OUT/'llm_external_run_status.json',{'status':'RESPONSES_AND_CPU_SCORING_COMPLETE','responses':60,'score_rows':60,'execution_guard_sha256':guard_doc['execution_guard_sha256'],'response_execution_guard_sha256':response_freeze['response_execution_guard_sha256'],'resume_ledger_sha256':sha(OUT/'llm_resume_ledger.json'),'response_record_hashes':{p.name:sha(p) for p in record_paths},'record_request_validation_association':'PASS_60_OF_60','scores_sha256':sha(score_path),'summary_sha256':sha(OUT/'llm_answer_summary.json'),'usage_cost_sha256':sha(OUT/'llm_usage_cost.json'),'external_requests_current_scoring_run':0})
    print({'llm_factual_scoring':'PASS','rows':60,'decision':decision,'wlt':[wins,losses,ties]})

{'llm_factual_scoring': 'PASS', 'rows': 60, 'decision': 'holdout_failed', 'wlt': [4, 2, 24]}
